# AI Predictive Maintenance System — ML Pipeline (NASA C-MAPSS FD001)

**Notebook scope:** the AI/ML component only — data loading, validation, EDA, feature engineering, failure prediction, calibration, anomaly detection, a deterministic Machine Health Indicator, explainability, evaluation, artifact export and an artifact reload test.

## 1. Project Introduction

**What is predictive maintenance?** Predictive maintenance (PdM) uses historical and current machine data to estimate *when* a machine is likely to need attention, so that maintenance can be planned rather than reactive.

**What does this project do?** It is a 100 % software project (no hardware, IoT devices or live sensors). The ML component answers, for every machine and operating cycle:

> *"Will this machine experience failure within the next H operating cycles?"*

and also produces an anomaly score, a deterministic **Machine Health Indicator (0–100)**, a prediction explanation, a data-quality status and a rule-based, model-assisted maintenance recommendation.

**Why C-MAPSS?** NASA C-MAPSS is a public **simulated turbofan-engine run-to-failure benchmark**. It provides full trajectories from healthy operation to failure, which is exactly what is needed to build and evaluate a failure-horizon model honestly. Important caveats:

* It is **simulated** data, not real field data.
* Its columns `sensor_1 … sensor_21` are **anonymized**. This notebook never assigns physical meanings to them.
* It is **not** motor / pump / compressor data. The application is dataset-agnostic; C-MAPSS is only the primary MVP demonstration dataset.

**What will this notebook train?**

| Component | Method |
|---|---|
| Failure prediction | Classical ML (XGBoost if available, else HistGradientBoosting; Logistic Regression baseline) |
| Probability calibration | Platt (sigmoid) / isotonic / none — chosen on validation data |
| Anomaly detection | Isolation Forest (+ simple z-score baseline for comparison) |
| Machine Health Indicator | Deterministic formula (**not** a trained model) |
| Explanations | TreeSHAP / model-native contributions |
| Recommendations | Transparent rules (**no LLM**) |

**What will FastAPI consume?** Saved artifacts under `artifacts/` (models, preprocessing, metadata JSON, evaluation, explanations) plus a small importable inference package (`artifacts/code/`).

> **This notebook trains and evaluates the ML models. The production web application will perform inference using the saved artifacts.**

Training and inference are kept separate: training-only logic lives in notebook cells; everything the backend needs at prediction time is written to importable modules in `artifacts/code/`.

## 2. Environment Setup

Only required packages are imported. `xgboost` and `shap` are optional: if they cannot be installed, the notebook falls back to `HistGradientBoostingClassifier` and model-native explanations.

In [ ]:
import importlib.util, subprocess, sys

def ensure_package(pip_name, import_name=None):
    """Install a package only if it is missing (Colab already has most of them)."""
    import_name = import_name or pip_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"{pip_name}: already available")

for _pip, _imp in [("xgboost", "xgboost"), ("shap", "shap")]:
    try:
        ensure_package(_pip, _imp)
    except Exception as exc:  # optional packages
        print(f"Optional package {_pip} could not be installed ({exc}). Fallbacks will be used.")

In [ ]:
import json, random, warnings, hashlib, platform, importlib, textwrap, os, shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import scipy
from scipy import stats
import sklearn
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, roc_curve, precision_recall_curve,
                             fbeta_score)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception:
    xgb, XGB_AVAILABLE = None, False
try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    shap, SHAP_AVAILABLE = None, False

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | scipy {scipy.__version__}")
print("xgboost available:", XGB_AVAILABLE, "| shap available:", SHAP_AVAILABLE)

### 2.1 Configuration

All tunable values live here. **Nothing below hard-codes paths or thresholds.** `FAILURE_HORIZON_H` is measured in *operating cycles* (C-MAPSS has no wall-clock time).

In [ ]:
IN_COLAB = "google.colab" in sys.modules

# ---- Data location -----------------------------------------------------------
DATASET_ID = "FD001"
DATASET_PATH = "/content/cmapss_data" if IN_COLAB else "./cmapss_data"
ALLOW_DOWNLOAD = False            # never download silently; opt in explicitly
DATASET_URL_BASE = None           # e.g. "https://your-host/cmapss" (folder containing the 3 files)
PROMPT_UPLOAD_IF_MISSING = True   # in Colab: show an upload dialog if files are missing
ARTIFACT_DIR = Path("/content/artifacts" if IN_COLAB else "./artifacts")

# ---- Kaggle auto-detection: data lives (read-only) under /kaggle/input, outputs must go to /kaggle/working ----
IN_KAGGLE = Path("/kaggle/input").exists()
if IN_KAGGLE:
    _found = sorted(Path("/kaggle/input").rglob(f"train_{DATASET_ID}.txt"))
    if _found:
        DATASET_PATH = str(_found[0].parent)
        print("Kaggle: using dataset folder", DATASET_PATH)
    else:
        print("Kaggle: train_FD001.txt not found under /kaggle/input. Add your dataset via 'Add Data' (right sidebar) "
              "and make sure it contains train_FD001.txt, test_FD001.txt and RUL_FD001.txt (a .zip must be extracted).")
    ARTIFACT_DIR = Path("/kaggle/working/artifacts")

# ---- Reproducibility & splitting ---------------------------------------------
RANDOM_STATE = 42
FAILURE_HORIZON_H = 30            # "failure within the next H operating cycles"
TEST_SIZE = 0.20                  # fraction of run-to-failure UNITS held out for test
VALIDATION_SIZE = 0.20            # fraction of UNITS for validation (selection/calibration/threshold)

# ---- Feature engineering -----------------------------------------------------
ROLLING_WINDOWS = (5, 10)
SLOPE_WINDOW = 10
PCT_CHANGE_EPS = 1e-6
PCT_CHANGE_CLIP = 1.0
LOW_VARIANCE_MAX_UNIQUE = 3               # <= this many distinct values -> "low variance"
LOW_VARIANCE_DOMINANT_FRACTION = 0.995    # one value covers >= this share of rows -> "low variance"

# ---- Modelling ---------------------------------------------------------------
IMBALANCE_WEIGHT_TRIGGER = 0.10           # positive rate below this -> use class weighting
SELECTION_PR_AUC_TOLERANCE = 0.005        # models within this PR-AUC of the best are tie-broken by Brier
CALIBRATION_METHODS = ("none", "sigmoid", "isotonic")
CALIBRATION_BRIER_TOLERANCE = 1e-4        # prefer simpler calibrator unless it improves Brier by more than this
CALIBRATION_CV_SPLITS = 5

# ---- Anomaly detection -------------------------------------------------------
ANOMALY_CONTAMINATION = 0.02              # share of HEALTHY training rows allowed to be flagged
HEALTHY_MIN_RUL = 100                     # rows with RUL >= this are treated as "healthy" for training the detector
IF_N_ESTIMATORS = 300

# ---- Threshold selection (configurable validation objective) ------------------
THRESHOLD_OBJECTIVE = "fbeta"             # "fbeta" or "precision_floor"
THRESHOLD_BETA = 2.0                      # beta>1 weights recall more than precision
THRESHOLD_MIN_PRECISION = 0.50            # used by "precision_floor": maximise recall s.t. precision >= this
THRESHOLD_GRID = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

# ---- Machine Health Indicator (deterministic, NOT a trained model) -------------
# HI = clip( 100 * (1 - P_fail) * (1 - anomaly_weight * anomaly_severity) - data_quality_penalty , 0, 100)
HEALTH_CONFIG = {
    "anomaly_weight": 0.30,
    "data_quality_penalty": {"DATA_OK": 0.0, "DATA_WARNING": 10.0},
}

# ---- Rule-based recommendation --------------------------------------------------
RECOMMENDATION_CONFIG = {
    "urgent_probability": 0.80,
    "urgent_health_max": 25.0,
    "schedule_health_max": 50.0,
    "inspect_health_max": 75.0,
    "inspect_probability_fraction_of_threshold": 0.5,
    "label": "AI recommendation - not a confirmed diagnosis",
    "disclaimer": ("This is an AI-generated recommendation derived from model outputs by transparent rules. "
                   "It is not a confirmed diagnosis or a maintenance instruction; a qualified person should make the final decision."),
}

# ---- Inference-time data-quality rules --------------------------------------------
QUALITY_CONFIG = {
    "range_margin_fraction": 0.5,        # training range is widened by this fraction on each side
    "extreme_invalid_fraction": 0.25,    # >= this share of features out of range -> DATA_INVALID
    "max_missing_fraction_latest": 0.20, # > this share of required values missing in latest row -> DATA_INVALID
    "min_history_invalid": 2,
}
TOP_K_FEATURES = 5

# ---- Documentation / exports ------------------------------------------------------------
FAILURE_HORIZON_RATIONALE = ("H is a configured planning lead time (in operating cycles) chosen before modelling as a business "
                             "parameter. It was NOT optimised on validation or test performance, and no sensitivity study over other "
                             "H values was run. Re-run the notebook with another FAILURE_HORIZON_H to compare.")
DEMO_SEED = 2024            # demo engines are drawn from the internal-test units with this fixed seed (never by model output)
N_DEMO_UNITS = 5
CURVE_MAX_POINTS = 200      # max points per curve in evaluation/curves.json
CALIBRATION_BINS = 10

# ---- Versioning ------------------------------------------------------------------
_now = datetime.now(timezone.utc)
TRAINING_TIMESTAMP = _now.isoformat(timespec="seconds")
MODEL_VERSION = f"cmapss-{DATASET_ID.lower()}-h{FAILURE_HORIZON_H}-{_now.strftime('%Y%m%dT%H%M%SZ')}"

assert 0 < TEST_SIZE + VALIDATION_SIZE < 1, "TEST_SIZE + VALIDATION_SIZE must be in (0, 1)"
assert THRESHOLD_OBJECTIVE in ("fbeta", "precision_floor")

# ---- Reproducibility ----------------------------------------------------------------
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ---- Artifact directories ----------------------------------------------------------
DATASET_DIR = Path(DATASET_PATH)
for _d in ["model", "preprocessing", "evaluation", "metadata", "explanations", "code"]:
    (ARTIFACT_DIR / _d).mkdir(parents=True, exist_ok=True)
CODE_DIR = ARTIFACT_DIR / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

def write_module(name, source):
    """Write reusable (inference-safe) code to artifacts/code/<name>.py and import it.
    Everything the backend needs at prediction time lives in these modules, not in notebook cells."""
    path = CODE_DIR / f"{name}.py"
    path.write_text(textwrap.dedent(source).lstrip("\n"))
    importlib.invalidate_caches()
    if name in sys.modules:
        return importlib.reload(sys.modules[name])
    return importlib.import_module(name)

def savefig(name):
    path = ARTIFACT_DIR / "evaluation" / f"{name}.png"
    plt.savefig(path, bbox_inches="tight", dpi=120)
    return path

def to_jsonable(obj):
    """Recursively convert numpy / pandas objects to JSON-safe Python types (NaN -> None)."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        return None if (np.isnan(obj) or np.isinf(obj)) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (pd.Series,)):
        return to_jsonable(obj.to_dict())
    if isinstance(obj, (pd.Timestamp, datetime, Path)):
        return str(obj)
    return obj

print("Model version:", MODEL_VERSION)
print("Artifact dir :", ARTIFACT_DIR.resolve())

## 3. Dataset Acquisition / Loading

**Expected C-MAPSS FD001 files** (plain text, whitespace separated, no header, 26 columns):

| File | Content |
|---|---|
| `train_FD001.txt` | Full run-to-failure trajectories (each unit runs until failure) |
| `test_FD001.txt` | Trajectories that **stop before failure** |
| `RUL_FD001.txt` | One number per test unit: true remaining cycles after its last observed cycle |

Columns: `unit_id, cycle, operating_setting_1..3, sensor_1..sensor_21`. The sensors are **anonymized**; no physical meaning is assigned here.

The data comes from the NASA Prognostics Center of Excellence repository (*Turbofan Engine Degradation Simulation*, Saxena et al., 2008). The notebook **never downloads anything silently**. Options:

* **A.** Put the three files in `DATASET_PATH` (e.g. upload them to Colab or mount Drive).
* **B.** Set `DATASET_URL_BASE` to a location you trust **and** set `ALLOW_DOWNLOAD = True`.
* **C.** In Colab, the cell below will offer an explicit upload dialog if files are missing (a `.zip` containing them is accepted).

Note: `unit_id` values in the train and test files overlap (both start at 1) but they are *different engines*. They are never concatenated in this notebook.

In [ ]:
OP_SETTING_COLS = [f"operating_setting_{i}" for i in (1, 2, 3)]
SENSOR_COLS = [f"sensor_{i}" for i in range(1, 22)]     # anonymized - do NOT rename
COLUMN_NAMES = ["unit_id", "cycle"] + OP_SETTING_COLS + SENSOR_COLS
REQUIRED_FILES = {
    "train": f"train_{DATASET_ID}.txt",
    "test": f"test_{DATASET_ID}.txt",
    "rul": f"RUL_{DATASET_ID}.txt",
}

def _missing_files(directory):
    return [fn for fn in REQUIRED_FILES.values() if not (directory / fn).exists()]

def _download_files(directory):
    import urllib.request
    for fn in _missing_files(directory):
        url = f"{DATASET_URL_BASE.rstrip('/')}/{fn}"
        print(f"Downloading (explicitly enabled): {url}")
        urllib.request.urlretrieve(url, directory / fn)

def _colab_upload(directory):
    from google.colab import files
    import zipfile
    print("Please select train_FD001.txt, test_FD001.txt and RUL_FD001.txt (or a .zip containing them).")
    for name, content in files.upload().items():
        (directory / name).write_bytes(content)
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(directory / name) as zf:
                for member in zf.namelist():
                    if Path(member).name in REQUIRED_FILES.values():
                        (directory / Path(member).name).write_bytes(zf.read(member))

def ensure_dataset_files(directory):
    directory.mkdir(parents=True, exist_ok=True)
    if _missing_files(directory) and ALLOW_DOWNLOAD and DATASET_URL_BASE:
        _download_files(directory)
    if _missing_files(directory) and IN_COLAB and PROMPT_UPLOAD_IF_MISSING:
        _colab_upload(directory)
    still_missing = _missing_files(directory)
    if still_missing:
        raise FileNotFoundError(
            f"Missing C-MAPSS files in {directory.resolve()}: {still_missing}. "
            "Upload them, point DATASET_PATH at them, or enable ALLOW_DOWNLOAD with a DATASET_URL_BASE you trust. "
            "On Kaggle: use 'Add Data' to attach a dataset containing the three files (they appear under /kaggle/input).")

def read_cmapss_table(path):
    df = pd.read_csv(path, sep=r"\s+", header=None)
    if df.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"{path.name}: expected {len(COLUMN_NAMES)} columns, found {df.shape[1]}. "
                         "Is this really a C-MAPSS file?")
    df.columns = COLUMN_NAMES
    df["unit_id"] = df["unit_id"].astype(int)
    df["cycle"] = df["cycle"].astype(int)
    return df

def load_cmapss(directory):
    ensure_dataset_files(directory)
    train = read_cmapss_table(directory / REQUIRED_FILES["train"])
    test = read_cmapss_table(directory / REQUIRED_FILES["test"])
    rul = pd.read_csv(directory / REQUIRED_FILES["rul"], sep=r"\s+", header=None)
    if rul.shape[1] != 1:
        raise ValueError("RUL file must contain exactly one column.")
    rul_test = pd.Series(rul.iloc[:, 0].to_numpy(), index=np.arange(1, len(rul) + 1), name="rul_after_last_cycle")
    rul_test.index.name = "unit_id"
    return train, test, rul_test

train_raw, test_raw, rul_test = load_cmapss(DATASET_DIR)
print("train:", train_raw.shape, "| test:", test_raw.shape, "| RUL entries:", len(rul_test))
display(train_raw.head(3))

## 4. Data Validation

Reusable validation functions check schema, dtypes, missing values, duplicates, unit IDs, cycle numbers, constant / extremely low-variance columns, and train/test consistency. "Low variance" is defined by **distinct-value count / dominant-value share** (not by relative standard deviation, which would wrongly discard small-but-real drifts in large-magnitude columns).

In [ ]:
def check_schema(df):
    missing = [c for c in COLUMN_NAMES if c not in df.columns]
    unexpected = [c for c in df.columns if c not in COLUMN_NAMES]
    return {"passed": not missing and not unexpected and list(df.columns) == COLUMN_NAMES,
            "missing_columns": missing, "unexpected_columns": unexpected}

def check_dtypes(df):
    non_numeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    return {"passed": not non_numeric, "non_numeric_columns": non_numeric}

def check_missing(df):
    per_col = df.isna().sum()
    return {"passed": int(per_col.sum()) == 0, "total_missing": int(per_col.sum()),
            "columns_with_missing": per_col[per_col > 0].to_dict()}

def check_duplicates(df):
    full = int(df.duplicated().sum())
    key = int(df.duplicated(subset=["unit_id", "cycle"]).sum())
    return {"passed": full == 0 and key == 0, "duplicate_rows": full, "duplicate_unit_cycle_keys": key}

def check_ids_and_cycles(df):
    g = df.groupby("unit_id")["cycle"]
    unit_ok = bool((df["unit_id"] > 0).all() and (df["unit_id"] % 1 == 0).all())
    cycle_ok = bool((df["cycle"] > 0).all() and (df["cycle"] % 1 == 0).all())
    contiguous = bool(((g.min() == 1) & (g.max() == g.count())).all())
    ordered = bool(g.apply(lambda s: s.is_monotonic_increasing).all())
    return {"passed": unit_ok and cycle_ok and contiguous and ordered,
            "valid_unit_ids": unit_ok, "valid_cycles": cycle_ok,
            "cycles_start_at_1_and_contiguous": contiguous, "cycles_sorted_within_unit": ordered}

def profile_sensor_variance(df, cols):
    rows = {}
    for c in cols:
        vc = df[c].round(6).value_counts(normalize=True)
        n_unique = int(df[c].nunique())
        dominant = float(vc.iloc[0]) if len(vc) else np.nan
        rows[c] = {"std": float(df[c].std()), "n_unique": n_unique, "dominant_fraction": dominant,
                   "is_constant": n_unique <= 1,
                   "is_low_variance": (n_unique <= LOW_VARIANCE_MAX_UNIQUE) or (dominant >= LOW_VARIANCE_DOMINANT_FRACTION)}
    return pd.DataFrame(rows).T.astype({"std": float, "n_unique": int, "dominant_fraction": float,
                                        "is_constant": bool, "is_low_variance": bool})

def check_train_test_consistency(train, test, rul):
    n_test_units = test["unit_id"].nunique()
    return {"passed": list(train.columns) == list(test.columns) and len(rul) == n_test_units
                      and bool((rul >= 0).all()) and not rul.isna().any(),
            "same_columns": list(train.columns) == list(test.columns),
            "rul_entries_match_test_units": len(rul) == n_test_units,
            "rul_non_negative_and_complete": bool((rul >= 0).all()) and not rul.isna().any()}

def build_quality_report(train, test, rul):
    report = {}
    for name, df in [("train", train), ("test", test)]:
        report[name] = {"shape": df.shape, "schema": check_schema(df), "dtypes": check_dtypes(df),
                        "missing": check_missing(df), "duplicates": check_duplicates(df),
                        "ids_cycles": check_ids_and_cycles(df)}
    report["consistency"] = check_train_test_consistency(train, test, rul)
    report["variance"] = profile_sensor_variance(train, OP_SETTING_COLS + SENSOR_COLS)
    return report

def print_quality_report(report):
    ok = lambda b: "PASS" if b else "FAIL"
    print("=== DATA QUALITY REPORT ===")
    for name in ("train", "test"):
        r = report[name]
        print(f"[{name}] shape={r['shape']}")
        for check in ("schema", "dtypes", "missing", "duplicates", "ids_cycles"):
            print(f"   {check:<11}: {ok(r[check]['passed'])}")
    print(f"[train/test consistency]: {ok(report['consistency']['passed'])}")
    var = report["variance"]
    print(f"[constant columns      ] {var.index[var.is_constant].tolist()}")
    print(f"[low-variance (not const)] {var.index[var.is_low_variance & ~var.is_constant].tolist()}")
    print("(Variance findings here are informational; columns to drop are decided on TRAINING units only in Section 9.)")

quality_report = build_quality_report(train_raw, test_raw, rul_test)
print_quality_report(quality_report)
hard_fail = [f"{n}.{c}" for n in ("train", "test") for c in ("schema", "dtypes", "missing", "duplicates", "ids_cycles")
             if not quality_report[n][c]["passed"]] + (["consistency"] if not quality_report["consistency"]["passed"] else [])
if hard_fail:
    print("FAILED CHECKS AND DETAILS:")
    for item in hard_fail:
        if item == "consistency":
            print(" - consistency:", quality_report["consistency"])
        else:
            n, c = item.split("."); print(f" - {item}:", quality_report[n][c])
    print("\nThe data is not valid C-MAPSS. Typical causes: the deliberately corrupted 'INVALID' test dataset, NaNs, duplicated rows, "
          "or files with a different layout. Upload a valid dataset and re-run from Section 3.")
assert not hard_fail, f"Data validation failed: {hard_fail}"
print("\nAll hard validation checks passed.")

## 5. Exploratory Data Analysis

EDA here is **descriptive only**. The full training file is used to understand the data, but no modelling decision (feature drop, scaling, selection) is learned from it — those are fitted on training *units* only in Section 9.

In [ ]:
train_eda = train_raw.copy()
cycles_per_unit = train_eda.groupby("unit_id")["cycle"].max()
test_last_cycle = test_raw.groupby("unit_id")["cycle"].max()

print(f"Train rows: {len(train_eda):,} | units: {train_eda.unit_id.nunique()} | columns: {train_eda.shape[1]}")
print(f"Test  rows: {len(test_raw):,} | units: {test_raw.unit_id.nunique()}")
print("\nTrain life length (cycles) per unit:")
display(cycles_per_unit.describe().round(1).to_frame("cycles"))

variance_profile = profile_sensor_variance(train_eda, OP_SETTING_COLS + SENSOR_COLS)
EDA_ACTIVE_SENSORS = [c for c in SENSOR_COLS if not variance_profile.loc[c, "is_low_variance"]]
print(f"\nSensors with usable variation (EDA view, full train file): {len(EDA_ACTIVE_SENSORS)} of {len(SENSOR_COLS)}")
display(variance_profile.sort_values("std", ascending=False).round(4))

**How to read the next figure:** train units run to failure with very different lifetimes; test units are cut off earlier at a random point. A wide spread of lifetimes means "cycle number" alone is not a reliable clock for failure, so the model must read degradation from sensor behaviour.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(cycles_per_unit, bins=20, ax=axes[0], color="steelblue")
axes[0].set_title("Train: total cycles until failure, per unit"); axes[0].set_xlabel("cycles")
sns.histplot(test_last_cycle, bins=20, ax=axes[1], color="darkorange")
axes[1].set_title("Test: last observed cycle, per unit (truncated)"); axes[1].set_xlabel("cycles")
plt.tight_layout(); plt.show()

In [ ]:
# Sensor distributions (active sensors only) and operating-condition distributions
n = len(EDA_ACTIVE_SENSORS); ncols = 4; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 2.6 * nrows))
for ax, c in zip(axes.ravel(), EDA_ACTIVE_SENSORS):
    sns.histplot(train_eda[c], bins=40, ax=ax, color="slateblue"); ax.set_title(c, fontsize=9); ax.set_xlabel("")
for ax in axes.ravel()[n:]: ax.axis("off")
plt.suptitle("Sensor value distributions (anonymized sensors, train file)", y=1.01); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, c in zip(axes, OP_SETTING_COLS):
    sns.histplot(train_eda[c], bins=40, ax=ax, color="seagreen"); ax.set_title(c, fontsize=9)
plt.suptitle("Operating-setting distributions", y=1.03); plt.tight_layout(); plt.show()

**Correlation matrix.** Strongly correlated sensor groups indicate redundant information. Redundancy is acceptable for tree models and regularised linear models, but it means individual feature importances should be interpreted with care (importance may be split between correlated features).

In [ ]:
corr = train_eda[EDA_ACTIVE_SENSORS].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True, cbar_kws={"label": "Pearson r"})
plt.title("Correlation between active sensors (train file)"); plt.tight_layout(); plt.show()

**Sensor trends across machine life.** Each panel shows the mean ± 1 std (across units) of a sensor versus *fraction of life elapsed* (0 = start, 1 = failure). Panels whose mean line drifts while staying separated from start-of-life noise carry degradation information.

In [ ]:
def life_normalised_profile(df, cols, n_bins=20):
    tmp = df[["unit_id", "cycle"] + cols].copy()
    tmp["life_fraction"] = tmp["cycle"] / tmp.groupby("unit_id")["cycle"].transform("max")
    tmp["bin"] = pd.cut(tmp["life_fraction"], bins=np.linspace(0, 1, n_bins + 1), include_lowest=True, labels=False)
    agg = tmp.groupby("bin")[cols].agg(["mean", "std"])
    agg.index = (agg.index + 0.5) / n_bins
    return agg

profile = life_normalised_profile(train_eda, EDA_ACTIVE_SENSORS)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 2.6 * nrows))
for ax, c in zip(axes.ravel(), EDA_ACTIVE_SENSORS):
    m, s = profile[(c, "mean")], profile[(c, "std")]
    ax.plot(m.index, m.values, color="crimson"); ax.fill_between(m.index, m - s, m + s, alpha=0.25, color="crimson")
    ax.set_title(c, fontsize=9)
for ax in axes.ravel()[n:]: ax.axis("off")
plt.suptitle("Sensor mean ± std vs fraction of life elapsed (0 = start, 1 = failure)", y=1.01)
plt.tight_layout(); plt.show()

## 6. Time-Series Understanding

The data is hierarchical: **`unit_id` → `cycle` → sensor readings**. Each unit is one independent trajectory. Consecutive rows of the same unit are strongly autocorrelated, so:

* rows are **not** independent samples → splits must be made by unit (Section 9);
* features must be computed **within a unit, using only current and past cycles** (Section 7).

**Which sensors carry degradation information?** We use statistical evidence, not guesses: for each active sensor we measure (a) the per-unit Spearman correlation between `cycle` and the sensor (monotonic drift), (b) how consistently the drift direction agrees across units, and (c) a standardized effect size between the first 20 % and last 20 % of life. This is **exploratory** (full train file) and is *not* used to select features for the model; it only supports interpretation. No physical meaning is attached to any sensor.

In [ ]:
def degradation_evidence(df, cols):
    life = df["cycle"] / df.groupby("unit_id")["cycle"].transform("max")
    early, late = df[life <= 0.2], df[life >= 0.8]
    rows = []
    for c in cols:
        rhos = []
        for _, g in df.groupby("unit_id"):
            rho, _p = stats.spearmanr(g["cycle"], g[c])
            if not np.isnan(rho): rhos.append(rho)
        rhos = np.array(rhos); med = float(np.median(rhos))
        pooled_sd = np.sqrt((early[c].var() + late[c].var()) / 2)
        rows.append({"sensor": c, "median_rho_with_cycle": med, "median_abs_rho": float(np.median(np.abs(rhos))),
                     "same_direction_fraction": float(np.mean(np.sign(rhos) == np.sign(med))),
                     "early_vs_late_effect_size": float((late[c].mean() - early[c].mean()) / pooled_sd) if pooled_sd > 0 else np.nan})
    return pd.DataFrame(rows).set_index("sensor").sort_values("median_abs_rho", ascending=False)

sensor_evidence = degradation_evidence(train_eda, EDA_ACTIVE_SENSORS)
display(sensor_evidence.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["crimson" if v > 0 else "steelblue" for v in sensor_evidence["median_rho_with_cycle"]]
ax.bar(sensor_evidence.index, sensor_evidence["median_rho_with_cycle"], color=colors)
ax.axhline(0, color="k", lw=0.6); ax.set_ylabel("median Spearman rho (cycle vs sensor)")
ax.set_title("Monotonic drift of each active sensor over life (red = rising, blue = falling)")
plt.xticks(rotation=60); plt.tight_layout(); plt.show()

**Example degradation trajectories.** Below: the four sensors with the strongest, most consistent drift, for three representative units (shortest, median and longest life). Note how noisy raw readings are and how lifetimes differ — this motivates smoothing (rolling means) and trend features.

In [ ]:
top_sensors = sensor_evidence.head(4).index.tolist()
order = cycles_per_unit.sort_values()
example_units = [order.index[0], order.index[len(order) // 2], order.index[-1]]
fig, axes = plt.subplots(len(top_sensors), 1, figsize=(10, 2.4 * len(top_sensors)), sharex=True)
for ax, s in zip(axes, top_sensors):
    for u in example_units:
        g = train_eda[train_eda.unit_id == u]
        ax.plot(g["cycle"], g[s], lw=0.9, label=f"unit {u} ({len(g)} cycles)")
    ax.set_ylabel(s)
axes[0].legend(ncol=3, fontsize=8); axes[-1].set_xlabel("cycle")
plt.suptitle("Raw sensor trajectories for three representative units", y=1.0); plt.tight_layout(); plt.show()

## 7. Feature Engineering

The feature builder is a **stateless, causal** transformer: every feature at cycle *t* uses only cycles ≤ *t* of the *same unit*. Because it learns no parameters it cannot leak information from validation/test; learned steps (dropping near-constant columns, median imputation) are fitted on training units only (Section 9).

| Feature group | Definition | Why |
|---|---|---|
| Raw sensors | `sensor_k` (only those with usable variance) | Current state |
| Operating settings | `operating_setting_k` (non-constant ones) | Operating context |
| Cycle number | `cycle` | Age of the machine |
| Rolling mean | `__roll_mean_w` (windows 5, 10; partial windows at start of life) | Noise reduction |
| Rolling std | `__roll_std_w` (ddof=0) | Volatility changes |
| Difference | `__diff1` = x_t − x_{t−1} (0 at first cycle) | Short-term change |
| Percentage change | `__pct_change1` = diff / previous value, guarded where |previous| ≤ eps, clipped to ±1 | Relative change, numerically safe |
| Trend / slope | `__slope_w` = least-squares slope over last w cycles (0 if < 2 points) | Recent drift direction |

The class is written to `artifacts/code/pdm_features.py` so that the backend can unpickle and reuse the **identical** code.

In [ ]:
FEATURES_SRC = r'''
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


def _slope(y):
    """Least-squares slope of y against 0..n-1 (NaN if <2 points or any NaN)."""
    n = len(y)
    if n < 2 or np.isnan(y).any():
        return np.nan
    t = np.arange(n, dtype=float)
    t -= t.mean()
    return float(np.dot(t, y) / np.dot(t, t))


class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Causal per-unit time-series feature builder (stateless: fit learns nothing).

    Input columns : cycle, setting_cols, sensor_cols and optionally unit_id.
    Every output row depends only on the same unit's current and PAST rows.
    """

    def __init__(self, sensor_cols, setting_cols, windows=(5, 10), slope_window=10,
                 pct_change_eps=1e-6, pct_change_clip=1.0):
        self.sensor_cols = sensor_cols
        self.setting_cols = setting_cols
        self.windows = windows
        self.slope_window = slope_window
        self.pct_change_eps = pct_change_eps
        self.pct_change_clip = pct_change_clip

    @property
    def max_lookback(self):
        """Number of most-recent rows needed to reproduce the features of the latest row exactly."""
        return int(max(max(self.windows), self.slope_window) + 1)

    def get_feature_names(self):
        names = ["cycle"] + list(self.setting_cols)
        for s in self.sensor_cols:
            names.append(s)
            names += [f"{s}__roll_mean_{w}" for w in self.windows]
            names += [f"{s}__roll_std_{w}" for w in self.windows]
            names += [f"{s}__diff1", f"{s}__pct_change1", f"{s}__slope_{self.slope_window}"]
        return names

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        settings, sensors = list(self.setting_cols), list(self.sensor_cols)
        missing = [c for c in ["cycle"] + settings + sensors if c not in X.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
        if not X.index.is_unique:
            raise ValueError("X.index must be unique")
        df = X[["cycle"] + settings + sensors].astype(float).copy()
        df["_unit"] = X["unit_id"].to_numpy() if "unit_id" in X.columns else 0
        df["_order"] = np.arange(len(df))
        df = df.sort_values(["_unit", "cycle", "_order"], kind="mergesort")
        grouped = df.groupby("_unit", sort=False)
        parts = {"cycle": df["cycle"]}
        for c in settings:
            parts[c] = df[c]
        for s in sensors:
            parts[s] = df[s]
            g = grouped[s]
            for w in self.windows:
                parts[f"{s}__roll_mean_{w}"] = g.transform(lambda v, w=w: v.rolling(w, min_periods=1).mean())
                parts[f"{s}__roll_std_{w}"] = g.transform(lambda v, w=w: v.rolling(w, min_periods=1).std(ddof=0))
            diff = g.diff()
            prev = g.shift(1)
            safe = prev.abs() > self.pct_change_eps
            pct = (diff / prev.where(safe)).clip(-self.pct_change_clip, self.pct_change_clip)
            parts[f"{s}__diff1"] = diff.fillna(0.0)
            parts[f"{s}__pct_change1"] = pct.fillna(0.0)
            sw = self.slope_window
            parts[f"{s}__slope_{sw}"] = g.transform(
                lambda v, sw=sw: v.rolling(sw, min_periods=2).apply(_slope, raw=True)).fillna(0.0)
        out = pd.DataFrame(parts, index=df.index)[self.get_feature_names()]
        return out.reindex(X.index)
'''
pdm_features = write_module("pdm_features", FEATURES_SRC)
from pdm_features import FeatureEngineer

**Causality (no-future-information) test.** We verify by experiment that (1) truncating a unit's history at cycle *k* leaves the features at cycle *k* unchanged, (2) corrupting all *future* rows does not change features at cycle *k*, and (3) using only the last `max_lookback` rows reproduces the latest row's features exactly (this is what the backend will rely on).

In [ ]:
_fe_test = FeatureEngineer(sensor_cols=EDA_ACTIVE_SENSORS, setting_cols=OP_SETTING_COLS[:2],
                           windows=tuple(ROLLING_WINDOWS), slope_window=SLOPE_WINDOW)
_unit = train_raw[train_raw.unit_id == 1].copy()
_full = _fe_test.transform(_unit)
_k = 50
_trunc = _fe_test.transform(_unit[_unit.cycle <= _k])
np.testing.assert_allclose(_full.loc[_trunc.index].to_numpy(), _trunc.to_numpy(), atol=1e-9)

_corrupt = _unit.copy()
_corrupt.loc[_corrupt.cycle > _k, EDA_ACTIVE_SENSORS] = 1e6          # wreck the future
_fc = _fe_test.transform(_corrupt)
np.testing.assert_allclose(_full[_full.index.isin(_unit.index[_unit.cycle <= _k])].to_numpy(),
                           _fc[_fc.index.isin(_unit.index[_unit.cycle <= _k])].to_numpy(), atol=1e-9)

_tail = _unit[_unit.cycle <= _k].tail(_fe_test.max_lookback)
np.testing.assert_allclose(_fe_test.transform(_tail).iloc[-1].to_numpy(), _full.loc[_trunc.index[-1]].to_numpy(), atol=1e-9)
print(f"Causality tests passed. Features per row: {_full.shape[1]} | max_lookback = {_fe_test.max_lookback}")

## 8. Failure Prediction Target

**Target:** `failure_within_h = 1` if the machine fails within the next `H = FAILURE_HORIZON_H` operating cycles, else `0`.

**Construction.** Let *RUL* (remaining useful life) at cycle *t* = (failure cycle) − *t*.

* **Training file (run-to-failure):** failure cycle = the unit's last cycle, so RUL = `max_cycle − cycle`. The final cycle has RUL = 0.
* **Test file (truncated):** the true remaining life after the last observed cycle is given by `RUL_FD001.txt`, so RUL at cycle *t* = `RUL_after_last_cycle + (last_observed_cycle − t)`.
* **Label:** `failure_within_h = (RUL ≤ H)`. The final H+1 cycles of each trajectory (RUL 0…H) are positives. Because every row has a known RUL, there is no censoring problem and no row is discarded.

`rul` and `failure_within_h` are **label-generation columns only**. They are kept out of the feature matrix, and Section 9 tests for leakage.

In [ ]:
def add_run_to_failure_labels(df, horizon, terminal_rul=None):
    """Attach `rul` and binary `failure_within_h`. terminal_rul: Series(unit_id -> RUL after last observed cycle)."""
    out = df.copy()
    last_cycle = out.groupby("unit_id")["cycle"].transform("max")
    if terminal_rul is None:
        out["rul"] = last_cycle - out["cycle"]
    else:
        out["rul"] = out["unit_id"].map(terminal_rul) + (last_cycle - out["cycle"])
    out["failure_within_h"] = (out["rul"] <= horizon).astype(int)
    return out

def class_distribution(y, name=""):
    y = pd.Series(y)
    pos = int(y.sum()); neg = int(len(y) - pos)
    return {"set": name, "rows": len(y), "negatives": neg, "positives": pos, "positive_rate": pos / len(y)}

train_labeled = add_run_to_failure_labels(train_raw, FAILURE_HORIZON_H)
test_labeled = add_run_to_failure_labels(test_raw, FAILURE_HORIZON_H, terminal_rul=rul_test)
display(pd.DataFrame([class_distribution(train_labeled.failure_within_h, "train file (all units)"),
                      class_distribution(test_labeled.failure_within_h, "official test file")]).round(4))

# Sanity: every training unit must end with positives, and the last row must have RUL == 0
assert (train_labeled.groupby("unit_id")["rul"].min() == 0).all()
assert train_labeled.groupby("unit_id").tail(1)["failure_within_h"].eq(1).all()

**Class imbalance.** Failure-within-H rows are a minority, and the exact rate depends on H. We apply an explicit, documented policy:

* If the positive rate of the *training* rows is below `IMBALANCE_WEIGHT_TRIGGER`, models use class weighting.
* Otherwise we **do not reweight**, because reweighting distorts predicted probabilities, and this project needs meaningful probabilities (Section 12). Imbalance is then handled by (a) choosing imbalance-aware metrics (PR-AUC, recall, F-beta), (b) unit-level splits, (c) probability calibration, and (d) an explicit, configurable decision threshold instead of the default 0.5.

## 9. Train / Validation / Test Split (by `unit_id`)

**Why unit-level splitting is mandatory.** Consecutive cycles of one machine are nearly identical. A random *row* split would place neighbouring cycles of the same machine in both train and test; the model could then "recognise" the machine instead of learning degradation, and every metric would be optimistically inflated. In deployment the model sees **machines it has never seen**, so evaluation must do the same.

* **Training units** – fit models, scaler/imputer, detectors.
* **Validation units** – model selection, calibration, threshold choice.
* **Internal test units** – full run-to-failure trajectories, touched only for final evaluation and the demo.
* **Official `test_FD001`** – a second, independent unseen evaluation (truncated trajectories, labels from `RUL_FD001.txt`).

In [ ]:
all_units = np.sort(train_labeled["unit_id"].unique())
holdout_frac = TEST_SIZE + VALIDATION_SIZE
train_units, holdout_units = train_test_split(all_units, test_size=holdout_frac, random_state=RANDOM_STATE)
val_units, test_units = train_test_split(holdout_units, test_size=TEST_SIZE / holdout_frac, random_state=RANDOM_STATE)
train_units, val_units, test_units = map(np.sort, (train_units, val_units, test_units))

assert not (set(train_units) & set(val_units)) and not (set(train_units) & set(test_units)) \
       and not (set(val_units) & set(test_units)), "unit leakage between splits!"
assert len(train_units) + len(val_units) + len(test_units) == len(all_units)

train_df = train_labeled[train_labeled.unit_id.isin(train_units)].copy()
val_df = train_labeled[train_labeled.unit_id.isin(val_units)].copy()
test_df = train_labeled[train_labeled.unit_id.isin(test_units)].copy()

split_table = pd.DataFrame([
    {**class_distribution(d.failure_within_h, n), "units": d.unit_id.nunique()}
    for n, d in [("train", train_df), ("validation", val_df), ("internal test", test_df), ("official test file", test_labeled)]
]).set_index("set")
display(split_table.round(4))

TRAIN_POSITIVE_RATE = float(train_df.failure_within_h.mean())
USE_CLASS_WEIGHTS = TRAIN_POSITIVE_RATE < IMBALANCE_WEIGHT_TRIGGER
print(f"Train positive rate = {TRAIN_POSITIVE_RATE:.3f} -> class weighting {'ENABLED' if USE_CLASS_WEIGHTS else 'NOT used (see Section 8 rationale)'}")

### 9.1 Fit learned preprocessing on TRAINING units only, then build matrices

Learned (data-dependent) steps, all fitted on `train_df` only:
1. dropping constant / low-variance sensors and operating settings,
2. median imputation (robustness for missing values at inference).

In [ ]:
sensor_profile_train = profile_sensor_variance(train_df, SENSOR_COLS)
setting_profile_train = profile_sensor_variance(train_df, OP_SETTING_COLS)
ACTIVE_SENSORS = sensor_profile_train.index[~sensor_profile_train.is_low_variance].tolist()
ACTIVE_SETTINGS = setting_profile_train.index[~setting_profile_train.is_low_variance].tolist()
DROPPED_SENSORS = [c for c in SENSOR_COLS if c not in ACTIVE_SENSORS]
DROPPED_SETTINGS = [c for c in OP_SETTING_COLS if c not in ACTIVE_SETTINGS]
print("Dropped sensors (constant/low variance in training units):", DROPPED_SENSORS)
print("Dropped operating settings:", DROPPED_SETTINGS)
print(f"Kept: {len(ACTIVE_SENSORS)} sensors, {len(ACTIVE_SETTINGS)} operating settings")

feature_engineer = FeatureEngineer(sensor_cols=ACTIVE_SENSORS, setting_cols=ACTIVE_SETTINGS,
                                   windows=tuple(ROLLING_WINDOWS), slope_window=SLOPE_WINDOW,
                                   pct_change_eps=PCT_CHANGE_EPS, pct_change_clip=PCT_CHANGE_CLIP)
FEATURE_NAMES = feature_engineer.get_feature_names()

X_train_raw = feature_engineer.transform(train_df)
imputer = SimpleImputer(strategy="median").fit(X_train_raw)          # fitted on TRAIN only

def impute_frame(F):
    return pd.DataFrame(imputer.transform(F), columns=FEATURE_NAMES, index=F.index)

def make_features(df):
    """raw dataframe -> imputed feature DataFrame (shared by all splits)."""
    return impute_frame(feature_engineer.transform(df))

X_train = impute_frame(X_train_raw)
X_val, X_test = make_features(val_df), make_features(test_df)
test_official_df = test_labeled
X_official = make_features(test_official_df)
y_train, y_val, y_test = train_df.failure_within_h.to_numpy(), val_df.failure_within_h.to_numpy(), test_df.failure_within_h.to_numpy()
y_official = test_official_df.failure_within_h.to_numpy()
print(f"Feature matrix: train {X_train.shape}, val {X_val.shape}, internal test {X_test.shape}, official test {X_official.shape}")

### 9.2 Target-leakage checks

We verify: (1) no label-generation column is a feature, (2) no single feature is (almost) a perfect predictor of the label, (3) features are causal (re-tested with the final engineer), and (4) a **negative control** — a model trained on *shuffled* labels — scores no better than chance on validation. If the pipeline leaked future/label information, the negative control would not collapse to chance.

In [ ]:
assert not any(n in ("rul", "failure_within_h", "unit_id") or "rul" in n.lower() for n in FEATURE_NAMES)
corr_with_label = X_train.apply(lambda col: abs(stats.spearmanr(col, y_train)[0])).fillna(0).sort_values(ascending=False)
print("Top |Spearman| of single features with the label (leak alarm if ~1.0):")
display(corr_with_label.head(5).round(3).to_frame("abs_spearman"))
assert corr_with_label.iloc[0] < 0.95, "A feature is suspiciously close to the label - investigate leakage"

_u = train_df[train_df.unit_id == train_units[0]]
_a = feature_engineer.transform(_u[_u.cycle <= 60]); _b = feature_engineer.transform(_u).loc[_a.index]
np.testing.assert_allclose(_a.to_numpy(), _b.to_numpy(), atol=1e-9)

_rng = np.random.RandomState(RANDOM_STATE)
_ctrl = HistGradientBoostingClassifier(max_iter=100, random_state=RANDOM_STATE).fit(X_train, _rng.permutation(y_train))
_ctrl_ap = average_precision_score(y_val, _ctrl.predict_proba(X_val)[:, 1])
print(f"Negative control (shuffled labels) validation PR-AUC = {_ctrl_ap:.3f}  vs prevalence = {y_val.mean():.3f}  (should be close)")
print("Leakage checks passed.")

## 10. Baseline Models

Candidates (all trained on training units only, evaluated on **validation units**):

1. **Cycle-only logistic regression** — a trivial reference: how much can "age" alone explain? (not eligible for selection)
2. **Logistic Regression** (scaled features) — interpretable linear baseline.
3. **HistGradientBoostingClassifier** — sklearn gradient boosting.
4. **XGBoost** (if available) — preferred gradient-boosted trees.

Metrics: ROC-AUC, **PR-AUC** (average precision), Brier score, and threshold-0.5 accuracy/precision/recall/F1 for reference. For rare-failure data, **PR-AUC and recall are more informative than accuracy**: a model predicting "no failure" always can still score high accuracy while catching zero failures, and ROC-AUC can look good when negatives dominate; PR-AUC focuses on the positive (failure) class.

In [ ]:
def ece_score(y_true, p, n_bins=10):
    """Expected calibration error with equal-width bins."""
    y_true, p = np.asarray(y_true), np.asarray(p)
    idx = np.clip(np.digitize(p, np.linspace(0, 1, n_bins + 1)[1:-1]), 0, n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            total += m.mean() * abs(y_true[m].mean() - p[m].mean())
    return float(total)

def evaluate_probs(y_true, p, threshold=0.5):
    y_true, p = np.asarray(y_true), np.asarray(p)
    pred = (p >= threshold).astype(int)
    both = len(np.unique(y_true)) == 2
    return {"accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_true, p) if both else np.nan,
            "pr_auc": average_precision_score(y_true, p) if both else np.nan,
            "brier": brier_score_loss(y_true, p), "ece": ece_score(y_true, p), "threshold": threshold}

def build_candidate_models(use_weights, pos_rate):
    models = {"logistic_regression": Pipeline([("scaler", StandardScaler()),
                ("clf", LogisticRegression(C=0.5, max_iter=3000, class_weight="balanced" if use_weights else None))]),
              "hist_gradient_boosting": HistGradientBoostingClassifier(
                max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=1.0, early_stopping=False,
                random_state=RANDOM_STATE)}
    if XGB_AVAILABLE:
        models["xgboost"] = xgb.XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
            min_child_weight=5, reg_lambda=1.0, tree_method="hist", eval_metric="logloss",
            scale_pos_weight=((1 - pos_rate) / pos_rate) if use_weights else 1.0,
            random_state=RANDOM_STATE, n_jobs=-1)
    return models

def fit_candidate(name, model, X, y, use_weights):
    if name == "hist_gradient_boosting" and use_weights:
        model.fit(X, y, sample_weight=compute_sample_weight("balanced", y))
    else:
        model.fit(X, y)
    return model

candidates = build_candidate_models(USE_CLASS_WEIGHTS, TRAIN_POSITIVE_RATE)
for _name, _m in candidates.items():
    fit_candidate(_name, _m, X_train, y_train, USE_CLASS_WEIGHTS)

ref_cycle_model = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression())]).fit(X_train[["cycle"]], y_train)
val_raw_probs = {"reference_cycle_only": ref_cycle_model.predict_proba(X_val[["cycle"]])[:, 1]}
val_raw_probs.update({n: m.predict_proba(X_val)[:, 1] for n, m in candidates.items()})

baseline_table = pd.DataFrame({n: evaluate_probs(y_val, p) for n, p in val_raw_probs.items()}).T
print(f"Validation prevalence (= PR-AUC of a random scorer): {y_val.mean():.3f}")
display(baseline_table[["roc_auc", "pr_auc", "brier", "ece", "accuracy", "precision", "recall", "f1"]].astype(float).round(4))

## 11. Failure Prediction Model — selection

**Selection rule (measurable, pre-declared):** among eligible candidates, keep those whose validation PR-AUC is within `SELECTION_PR_AUC_TOLERANCE` of the best, then pick the one with the **lowest validation Brier score** (probability quality). XGBoost is therefore chosen when it is competitive, not merely because it is preferred; HistGradientBoosting is the fallback when XGBoost is unavailable. The output of the final model is `P(failure within H cycles)` ∈ [0, 1].

In [ ]:
def select_primary_model(table, tol, reference_names=("reference_cycle_only",)):
    elig = table.drop(index=list(reference_names)).astype(float)
    best = elig["pr_auc"].max()
    contenders = elig[elig["pr_auc"] >= best - tol]
    return contenders["brier"].idxmin(), contenders

PRIMARY_NAME, _contenders = select_primary_model(baseline_table, SELECTION_PR_AUC_TOLERANCE)
print("Contenders within PR-AUC tolerance:", _contenders.index.tolist())
print("SELECTED failure model:", PRIMARY_NAME)
final_model = candidates[PRIMARY_NAME]
FINAL_MODEL_KIND = PRIMARY_NAME
_p = final_model.predict_proba(X_val)[:, 1]
assert _p.min() >= 0 and _p.max() <= 1
print(f"Raw (uncalibrated) probability range on validation: [{_p.min():.4f}, {_p.max():.4f}]")
display(baseline_table.loc[[PRIMARY_NAME], ["roc_auc", "pr_auc", "brier", "ece"]].astype(float).round(4))

## 12. Probability Calibration

Raw model scores are not guaranteed to be probabilities. We compare three options and choose **on validation data only**:

* `none` – use raw scores,
* `sigmoid` – Platt scaling (logistic regression on the logit of the raw score),
* `isotonic` – monotone non-parametric mapping (flexible, but can overfit on small sets).

Because the validation units are also used to *fit* the calibrator, choosing the method on in-sample predictions would be biased. We therefore compare methods with **grouped (by unit) out-of-fold calibration** on the validation set, choose the lowest Brier score (preferring the simpler method unless a richer one improves Brier by more than `CALIBRATION_BRIER_TOLERANCE`), and only then fit the final calibrator on all validation units. Calibration is a monotone mapping, so it does not change the ranking (for `sigmoid`/`none`). **Calibrated probabilities are estimates, not guarantees.**

In [ ]:
CALIBRATION_SRC = r'''
import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

_EPS = 1e-6


def _logit(p):
    p = np.clip(np.asarray(p, dtype=float), _EPS, 1 - _EPS)
    return np.log(p / (1 - p))


class ProbabilityCalibrator:
    """Maps raw model probabilities to calibrated probabilities. method: none | sigmoid | isotonic."""

    def __init__(self, method="sigmoid"):
        self.method = method

    def fit(self, raw_proba, y):
        raw = np.asarray(raw_proba, dtype=float).ravel()
        y = np.asarray(y).astype(int).ravel()
        if self.method == "sigmoid":
            self._model = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000).fit(_logit(raw).reshape(-1, 1), y)
        elif self.method == "isotonic":
            self._model = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(raw, y)
        elif self.method == "none":
            self._model = None
        else:
            raise ValueError(f"Unknown calibration method: {self.method}")
        return self

    def transform(self, raw_proba):
        raw = np.asarray(raw_proba, dtype=float).ravel()
        if self.method == "none":
            out = raw
        elif self.method == "sigmoid":
            out = self._model.predict_proba(_logit(raw).reshape(-1, 1))[:, 1]
        else:
            out = self._model.predict(raw)
        return np.clip(out, 0.0, 1.0)


class CalibratedFailureModel:
    """Base classifier + fitted calibrator. predict_proba(X)[:, 1] = calibrated P(failure within H)."""

    def __init__(self, base_model, calibrator, feature_names):
        self.base_model = base_model
        self.calibrator = calibrator
        self.feature_names = list(feature_names)

    def predict_raw_proba(self, X):
        return self.base_model.predict_proba(X)[:, 1]

    def predict_proba(self, X):
        p = self.calibrator.transform(self.predict_raw_proba(X))
        return np.column_stack([1 - p, p])
'''
pdm_calibration = write_module("pdm_calibration", CALIBRATION_SRC)
from pdm_calibration import ProbabilityCalibrator, CalibratedFailureModel

def oof_calibrated(raw_p, y, groups, method, n_splits):
    raw_p, y, groups = map(np.asarray, (raw_p, y, groups))
    out = np.zeros_like(raw_p, dtype=float)
    for tr, te in GroupKFold(n_splits=min(n_splits, len(np.unique(groups)))).split(raw_p, y, groups):
        out[te] = ProbabilityCalibrator(method).fit(raw_p[tr], y[tr]).transform(raw_p[te])
    return out

raw_val = final_model.predict_proba(X_val)[:, 1]
raw_test = final_model.predict_proba(X_test)[:, 1]
groups_val = val_df["unit_id"].to_numpy()

oof = {m: oof_calibrated(raw_val, y_val, groups_val, m, CALIBRATION_CV_SPLITS) for m in CALIBRATION_METHODS}
calib_compare = pd.DataFrame({m: evaluate_probs(y_val, p) for m, p in oof.items()}).T[["brier", "ece", "roc_auc", "pr_auc"]].astype(float)
display(calib_compare.round(5))

def choose_calibration(table, order=CALIBRATION_METHODS, tol=CALIBRATION_BRIER_TOLERANCE):
    best = order[0]
    for m in order[1:]:
        if table.loc[m, "brier"] < table.loc[best, "brier"] - tol:
            best = m
    return best

CALIBRATION_METHOD = choose_calibration(calib_compare)
print("Chosen calibration method:", CALIBRATION_METHOD)
calibrator = ProbabilityCalibrator(CALIBRATION_METHOD).fit(raw_val, y_val)
calibrated_model = CalibratedFailureModel(final_model, calibrator, FEATURE_NAMES)

p_val_cal_oof = oof[CALIBRATION_METHOD]                      # honest validation estimate (used for threshold selection)
p_test_cal = calibrated_model.predict_proba(X_test)[:, 1]
p_off_cal = calibrated_model.predict_proba(X_official)[:, 1]
p_off_raw = final_model.predict_proba(X_official)[:, 1]

before_after = pd.DataFrame({
    "validation raw (in-sample)": evaluate_probs(y_val, raw_val),
    f"validation {CALIBRATION_METHOD} (out-of-fold)": evaluate_probs(y_val, p_val_cal_oof),
    "internal test raw": evaluate_probs(y_test, raw_test),
    f"internal test {CALIBRATION_METHOD}": evaluate_probs(y_test, p_test_cal),
}).T[["brier", "ece", "roc_auc", "pr_auc"]].astype(float)
display(before_after.round(5))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, y_, raw_, cal_) in zip(axes, [("Validation (out-of-fold calibrated)", y_val, raw_val, p_val_cal_oof),
                                              ("Internal test", y_test, raw_test, p_test_cal)]):
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfect")
    for lab, pp in [("before (raw)", raw_), (f"after ({CALIBRATION_METHOD})", cal_)]:
        try:
            frac, mean_p = calibration_curve(y_, pp, n_bins=10, strategy="quantile")
            ax.plot(mean_p, frac, marker="o", label=lab)
        except ValueError:
            pass
    ax.set_xlabel("mean predicted probability"); ax.set_ylabel("observed failure frequency"); ax.set_title(title); ax.legend()
plt.tight_layout(); savefig("calibration_before_after"); plt.show()

## 13. Anomaly Detection (Isolation Forest)

**Method.** An Isolation Forest is trained **only on healthy-looking training observations** (training units, rows with `RUL ≥ HEALTHY_MIN_RUL`). It needs no failure labels, so it can also flag unusual behaviour that was never seen during a failure. Inputs: active raw sensors + their longest rolling mean (smoothing).

**Score semantics.** scikit-learn's `score_samples` returns the *negative* of the original Isolation-Forest anomaly score. We use `raw = −score_samples`, so **higher raw = more anomalous** (values lie roughly in (0, 1); ≈0.5 is typical, →1 is very isolated). The raw value is stored and exposed. For user-facing display we add two transparent derived quantities:

* `anomaly_score` ∈ [0, 1] = the **percentile of the raw score within the healthy reference distribution** (healthy validation rows) (0.97 means "more anomalous than 97 % of healthy training rows"). It is a rank, **not** a probability and not a probability of failure.
* `anomaly_flag` = raw score above the `(1 − ANOMALY_CONTAMINATION)` quantile of a **healthy reference set**. The forest is fitted on healthy *training* rows, but the reference/alarm threshold is **recalibrated on healthy rows of the validation units** (unseen by the forest): scores of the forest's own training rows look artificially normal, so an in-sample threshold would flag more than the target share of genuinely unseen healthy data. Both rates are measured and printed.
* `anomaly_severity` ∈ [0, 1] = how far past the flag percentile the score is (0 at/below the flag boundary, 1 at the top of the reference distribution). Used by the Machine Health Indicator.

An anomaly means "unusual compared with healthy training data", **not** "failure". A simple **z-score baseline** (RMS of per-feature z-scores relative to healthy data) is trained the same way for comparison.

In [ ]:
ANOMALY_SRC = r'''
import numpy as np
from sklearn.ensemble import IsolationForest


def _matrix(X, feature_names):
    return np.asarray(X[list(feature_names)], dtype=float)


class AnomalyScorer:
    """Isolation Forest trained on healthy observations, with transparent user-facing scores."""

    def __init__(self, feature_names, contamination=0.02, n_estimators=300, random_state=42):
        self.feature_names = list(feature_names)
        self.contamination = contamination
        self.n_estimators = n_estimators
        self.random_state = random_state

    def fit(self, X_healthy):
        self.iforest_ = IsolationForest(n_estimators=self.n_estimators, contamination="auto",
                                        random_state=self.random_state, n_jobs=-1)
        self.iforest_.fit(_matrix(X_healthy, self.feature_names))
        raw = self.raw_score(X_healthy)
        self.reference_scores_ = np.sort(raw)
        self.flag_percentile_ = 1.0 - self.contamination
        self.flag_threshold_ = float(np.quantile(raw, self.flag_percentile_))
        self.calibration_source_ = "training healthy rows (in-sample)"
        return self

    def recalibrate(self, X_reference):
        """Re-derive the percentile reference and alarm threshold from healthy rows the forest has NOT seen."""
        raw = self.raw_score(X_reference)
        self.reference_scores_ = np.sort(raw)
        self.flag_threshold_ = float(np.quantile(raw, self.flag_percentile_))
        self.calibration_source_ = "held-out healthy rows"
        return self

    def raw_score(self, X):
        """Higher = more anomalous (= -sklearn score_samples)."""
        return -self.iforest_.score_samples(_matrix(X, self.feature_names))

    def score(self, X):
        raw = self.raw_score(X)
        pct = np.searchsorted(self.reference_scores_, raw, side="right") / len(self.reference_scores_)
        q = self.flag_percentile_
        return {"raw": raw,
                "anomaly_score": pct,
                "anomaly_flag": raw > self.flag_threshold_,
                "anomaly_severity": np.clip((pct - q) / (1.0 - q), 0.0, 1.0)}


class ZScoreBaseline:
    """Simple statistical baseline: RMS of per-feature z-scores w.r.t. healthy data."""

    def __init__(self, feature_names, contamination=0.02):
        self.feature_names = list(feature_names)
        self.contamination = contamination

    def fit(self, X_healthy):
        M = _matrix(X_healthy, self.feature_names)
        self.mean_ = M.mean(axis=0)
        sd = M.std(axis=0)
        self.std_ = np.where(sd > 0, sd, 1.0)
        self.flag_threshold_ = float(np.quantile(self.raw_score(X_healthy), 1.0 - self.contamination))
        return self

    def recalibrate(self, X_reference):
        self.flag_threshold_ = float(np.quantile(self.raw_score(X_reference), 1.0 - self.contamination))
        return self

    def raw_score(self, X):
        z = (_matrix(X, self.feature_names) - self.mean_) / self.std_
        return np.sqrt((z ** 2).mean(axis=1))

    def score(self, X):
        raw = self.raw_score(X)
        return {"raw": raw, "anomaly_flag": raw > self.flag_threshold_}
'''
pdm_anomaly = write_module("pdm_anomaly", ANOMALY_SRC)
from pdm_anomaly import AnomalyScorer, ZScoreBaseline

ANOMALY_FEATURES = ACTIVE_SENSORS + [f"{s}__roll_mean_{max(ROLLING_WINDOWS)}" for s in ACTIVE_SENSORS]
healthy_mask = (train_df["rul"] >= HEALTHY_MIN_RUL).to_numpy()
X_healthy = X_train.loc[healthy_mask]
print(f"Healthy training observations for the anomaly detector: {len(X_healthy):,} ({healthy_mask.mean():.1%} of training rows)")

anomaly_scorer = AnomalyScorer(ANOMALY_FEATURES, contamination=ANOMALY_CONTAMINATION,
                               n_estimators=IF_N_ESTIMATORS, random_state=RANDOM_STATE).fit(X_healthy)
zscore_baseline = ZScoreBaseline(ANOMALY_FEATURES, contamination=ANOMALY_CONTAMINATION).fit(X_healthy)

# The alarm threshold fitted on the forest's own training rows is optimistic (in-sample scores look more normal), so unseen
# healthy data would be flagged more often than ANOMALY_CONTAMINATION. We measure that, then recalibrate the percentile
# reference and alarm threshold on HEALTHY rows of VALIDATION units (unseen by the forest; no failure labels needed).
X_healthy_val = X_val.loc[(val_df["rul"] >= HEALTHY_MIN_RUL).to_numpy()]
ANOMALY_FLAG_RATES = {"target_rate": ANOMALY_CONTAMINATION, "n_validation_healthy_rows": int(len(X_healthy_val)),
                      "isolation_forest": {
                          "train_healthy_in_sample": float(anomaly_scorer.score(X_healthy)["anomaly_flag"].mean()),
                          "validation_healthy_before_recalibration": float(anomaly_scorer.score(X_healthy_val)["anomaly_flag"].mean())}}
anomaly_scorer.recalibrate(X_healthy_val)
zscore_baseline.recalibrate(X_healthy_val)
ANOMALY_FLAG_RATES["isolation_forest"]["validation_healthy_after_recalibration"] = float(anomaly_scorer.score(X_healthy_val)["anomaly_flag"].mean())
print(f"Healthy flag rate (target {ANOMALY_CONTAMINATION:.1%}): in-sample {ANOMALY_FLAG_RATES['isolation_forest']['train_healthy_in_sample']:.2%} | "
      f"unseen validation healthy BEFORE recalibration {ANOMALY_FLAG_RATES['isolation_forest']['validation_healthy_before_recalibration']:.2%} | "
      f"AFTER {ANOMALY_FLAG_RATES['isolation_forest']['validation_healthy_after_recalibration']:.2%}")

**Evaluating the detector.** There are no anomaly labels in C-MAPSS, so we use a *proxy*: on unseen internal-test units, how well does the score separate **healthy** rows (`RUL ≥ HEALTHY_MIN_RUL`) from **near-failure** rows (`RUL ≤ H`)? This measures whether anomalies coincide with late-life behaviour; it is not a ground-truth anomaly evaluation.

In [ ]:
def anomaly_proxy_eval(X, rul, scorer, name):
    healthy = (rul >= HEALTHY_MIN_RUL).to_numpy(); near = (rul <= FAILURE_HORIZON_H).to_numpy()
    sel = healthy | near
    s = scorer.score(X); y = near[sel].astype(int)
    return {"method": name, "proxy_roc_auc": roc_auc_score(y, s["raw"][sel]),
            "flag_rate_healthy_rows": float(s["anomaly_flag"][healthy].mean()),
            "flag_rate_near_failure_rows": float(s["anomaly_flag"][near].mean())}

anomaly_eval = pd.DataFrame([anomaly_proxy_eval(X_test, test_df["rul"], anomaly_scorer, "IsolationForest"),
                             anomaly_proxy_eval(X_test, test_df["rul"], zscore_baseline, "z-score baseline (RMS)")]).set_index("method")
display(anomaly_eval.round(4))
ANOMALY_FLAG_RATES["isolation_forest"]["internal_test_healthy_rows"] = float(anomaly_eval.loc["IsolationForest", "flag_rate_healthy_rows"])
print(f"Healthy flag rate on unseen INTERNAL-TEST units (target {ANOMALY_CONTAMINATION:.1%}): {ANOMALY_FLAG_RATES['isolation_forest']['internal_test_healthy_rows']:.2%}")

_s = anomaly_scorer.score(X_test); _z = zscore_baseline.score(X_test)
_df = pd.DataFrame({"rul": test_df["rul"].to_numpy(), "if_score": _s["anomaly_score"], "if_flag": _s["anomaly_flag"], "z_flag": _z["anomaly_flag"]})
_df["rul_bin"] = (_df["rul"].clip(upper=200) // 10) * 10
_g = _df.groupby("rul_bin").mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(_g.index, _g["if_score"]); axes[0].invert_xaxis(); axes[0].set_xlabel("RUL (cycles, binned; failure at 0 ->)"); axes[0].set_ylabel("mean anomaly_score"); axes[0].set_title("Isolation Forest anomaly_score vs remaining life")
axes[1].plot(_g.index, _g["if_flag"], label="IsolationForest"); axes[1].plot(_g.index, _g["z_flag"], label="z-score baseline")
axes[1].invert_xaxis(); axes[1].set_xlabel("RUL (cycles, binned)"); axes[1].set_ylabel("flag rate"); axes[1].legend(); axes[1].set_title("Share of rows flagged as anomalous")
plt.tight_layout(); savefig("anomaly_vs_rul"); plt.show()

## 14. Machine Health Indicator (0–100)

The **Machine Health Indicator** is a **deterministic formula — not a trained ML model** and **not a physical measurement**. It condenses model outputs into one bounded number:

```
HI = clip( 100 × (1 − P_fail) × (1 − w_a × anomaly_severity) − data_quality_penalty , 0 , 100 )
```

* `P_fail` — calibrated probability of failure within H cycles (dominant term: HI → 0 as P_fail → 1).
* `anomaly_severity` ∈ [0,1] — Section 13. `w_a = anomaly_weight` is the **maximum share** of health an anomaly alone can remove (default 0.30), so anomaly alone can never drive HI to 0.
* `data_quality_penalty` — points subtracted for `DATA_WARNING` (default 10). For `DATA_INVALID`, **no HI is computed** (it would be meaningless).

All weights live in `HEALTH_CONFIG` and are saved with the artifacts. Bounded, deterministic, configurable.

**Additive decomposition (for UI breakdowns).** Because HI = 100 − 100·P − 100·(1−P)·w·s − penalty (before clipping), it splits exactly into additive point deductions: *failure risk* = 100·P, *anomaly* = 100·(1−P)·w·s, *data quality* = penalty. `health_breakdown()` returns these rows; a *trend* row is reported as **"not enabled"** (no trend term is used).

In [ ]:
HEALTH_SRC = r'''
import numpy as np

DEFAULT_HEALTH_CONFIG = {
    "anomaly_weight": 0.30,
    "data_quality_penalty": {"DATA_OK": 0.0, "DATA_WARNING": 10.0},
}


def compute_health_indicator(failure_probability, anomaly_severity, data_quality_status="DATA_OK", config=None):
    """Machine Health Indicator in [0, 100] (deterministic; NOT a trained model, NOT a physical measurement).

    HI = clip(100 * (1 - P_fail) * (1 - w_a * severity) - penalty, 0, 100).
    Returns None for DATA_INVALID. Accepts scalars or numpy arrays.
    """
    cfg = {**DEFAULT_HEALTH_CONFIG, **(config or {})}
    if data_quality_status == "DATA_INVALID":
        return None
    w_a = float(cfg["anomaly_weight"])
    if not 0.0 <= w_a <= 1.0:
        raise ValueError("anomaly_weight must be in [0, 1]")
    p = np.clip(np.asarray(failure_probability, dtype=float), 0.0, 1.0)
    s = np.clip(np.asarray(anomaly_severity, dtype=float), 0.0, 1.0)
    penalty = float(cfg["data_quality_penalty"].get(data_quality_status, 0.0))
    hi = np.clip(100.0 * (1.0 - p) * (1.0 - w_a * s) - penalty, 0.0, 100.0)
    return float(np.round(hi, 2)) if hi.ndim == 0 else np.round(hi, 2)


def health_breakdown(failure_probability, anomaly_severity, data_quality_status="DATA_OK", config=None):
    """Additive decomposition of the HI (scalar inputs): 100 - failure_risk - anomaly - data_quality (+clip) = HI."""
    cfg = {**DEFAULT_HEALTH_CONFIG, **(config or {})}
    if data_quality_status == "DATA_INVALID":
        return None
    p = float(np.clip(failure_probability, 0.0, 1.0))
    s = float(np.clip(anomaly_severity, 0.0, 1.0))
    w_a = float(cfg["anomaly_weight"])
    penalty = float(cfg["data_quality_penalty"].get(data_quality_status, 0.0))
    risk, anom = 100.0 * p, 100.0 * (1.0 - p) * w_a * s
    raw = 100.0 - risk - anom - penalty
    hi = float(np.clip(raw, 0.0, 100.0))
    return {"start": 100.0, "failure_risk_points": round(risk, 2), "anomaly_points": round(anom, 2),
            "data_quality_points": round(penalty, 2), "trend_points": None, "trend_status": "not_enabled",
            "clipping_adjustment_points": round(hi - raw, 2), "health_indicator": round(hi, 2)}
'''
pdm_health = write_module("pdm_health", HEALTH_SRC)
from pdm_health import compute_health_indicator, health_breakdown

grid_p = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]; grid_s = [0.0, 0.5, 1.0]
hi_table = pd.DataFrame({f"severity={s}": [compute_health_indicator(p, s, "DATA_OK", HEALTH_CONFIG) for p in grid_p] for s in grid_s}, index=[f"P_fail={p}" for p in grid_p])
print("Formula behaviour (DATA_OK):"); display(hi_table)
print("With DATA_WARNING, P_fail=0.1, severity=0.5 ->", compute_health_indicator(0.1, 0.5, "DATA_WARNING", HEALTH_CONFIG))
print("With DATA_INVALID ->", compute_health_indicator(0.1, 0.5, "DATA_INVALID", HEALTH_CONFIG))
print("Breakdown example (P=0.2, severity=0.6, DATA_WARNING):", health_breakdown(0.2, 0.6, "DATA_WARNING", HEALTH_CONFIG))
_rng = np.random.RandomState(0)
for _p, _s in _rng.rand(200, 2):                                   # decomposition identity check
    _b = health_breakdown(_p, _s, "DATA_WARNING", HEALTH_CONFIG)
    assert abs(_b["start"] - _b["failure_risk_points"] - _b["anomaly_points"] - _b["data_quality_points"] + _b["clipping_adjustment_points"] - _b["health_indicator"]) < 0.02
    assert abs(_b["health_indicator"] - compute_health_indicator(_p, _s, "DATA_WARNING", HEALTH_CONFIG)) < 1e-9
print("Additive decomposition verified.")

## 15. Explainability

**Local explanations** (per prediction) and **global importance** (whole model). For the gradient-boosted model we use **TreeSHAP** (XGBoost's built-in `pred_contribs`, the same algorithm as `shap.TreeExplainer`, so no heavy dependency is needed at inference time). Fallbacks: coefficient × standardised value for logistic regression, and feature-occlusion for any other model.

Reading a contribution: values are in **log-odds of the *uncalibrated* model**. **Positive → pushes toward "failure within H cycles"; negative → pushes away.** Calibration is a monotone mapping, so directions are preserved. Explanations describe the model's behaviour, not physical causes; `sensor_k` names remain anonymized and feature names like `sensor_k__slope_10` just mean "recent trend of anonymized sensor k".

In [ ]:
EXPLAIN_SRC = r'''
import numpy as np
import pandas as pd


def _logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def model_kind(model):
    if type(model).__name__ == "XGBClassifier":
        return "xgboost"
    steps = getattr(model, "steps", None)
    if steps and type(steps[-1][1]).__name__ == "LogisticRegression":
        return "logistic_regression"
    return "generic"


def raw_contributions(model, x_row, feature_names, reference_values):
    """Per-feature contributions (log-odds scale, uncalibrated model) for a single-row DataFrame."""
    kind = model_kind(model)
    x = x_row[list(feature_names)]
    if kind == "xgboost":
        import xgboost as xgb
        contrib = model.get_booster().predict(xgb.DMatrix(x), pred_contribs=True)[0][:-1]
        return np.asarray(contrib, dtype=float), "TreeSHAP (xgboost pred_contribs)"
    if kind == "logistic_regression":
        scaler, clf = model.steps[0][1], model.steps[-1][1]
        return clf.coef_[0] * scaler.transform(x)[0], "logistic coefficient x standardised value"
    base = _logit(model.predict_proba(x)[0, 1])
    occl = pd.concat([x] * len(feature_names), ignore_index=True)
    for i, f in enumerate(feature_names):
        occl.iloc[i, occl.columns.get_loc(f)] = reference_values[f]
    return base - _logit(model.predict_proba(occl)[:, 1]), "occlusion vs training-median reference"


def explain_row(model, x_row, feature_names, reference_values, top_k=5):
    contrib, method = raw_contributions(model, x_row, feature_names, reference_values)
    order = np.argsort(-np.abs(contrib))[:top_k]
    top = [{"feature": feature_names[i],
            "value": float(x_row[feature_names[i]].iloc[0]),
            "contribution": float(contrib[i]),
            "direction": "increases_failure_risk" if contrib[i] > 0 else "decreases_failure_risk"}
           for i in order]
    return {"method": method, "space": "log-odds of uncalibrated model", "top_features": top}
'''
pdm_explain = write_module("pdm_explain", EXPLAIN_SRC)
from pdm_explain import explain_row, model_kind

reference_values = pd.Series(imputer.statistics_, index=FEATURE_NAMES)
X_sample = X_test.sample(min(1500, len(X_test)), random_state=RANDOM_STATE)
kind = model_kind(final_model)

if kind == "xgboost":
    sv = final_model.get_booster().predict(xgb.DMatrix(X_sample), pred_contribs=True)[:, :-1]
    global_importance = pd.Series(np.abs(sv).mean(axis=0), index=FEATURE_NAMES).sort_values(ascending=False)
    GLOBAL_IMPORTANCE_METHOD = "mean |TreeSHAP| on internal-test sample (log-odds)"
    if SHAP_AVAILABLE:
        try:
            plt.figure(); shap.summary_plot(sv, X_sample, max_display=15, show=False)
            plt.title("SHAP summary (uncalibrated model, internal-test sample)")
            plt.savefig(ARTIFACT_DIR / "explanations" / "shap_summary.png", bbox_inches="tight", dpi=120)
            savefig("shap_summary"); plt.show()
        except Exception as exc:
            print("SHAP summary plot skipped:", exc); plt.close("all")
else:
    pi = permutation_importance(final_model, X_val, y_val, scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
    global_importance = pd.Series(pi.importances_mean, index=FEATURE_NAMES).sort_values(ascending=False)
    GLOBAL_IMPORTANCE_METHOD = "permutation importance (PR-AUC drop) on validation units"
    print("SHAP summary plot not produced for this model type; permutation importance used instead.")

print("Global importance method:", GLOBAL_IMPORTANCE_METHOD)
fig, ax = plt.subplots(figsize=(7, 5))
global_importance.head(15)[::-1].plot.barh(ax=ax, color="teal"); ax.set_title("Top 15 features (global importance)")
plt.tight_layout(); savefig("feature_importance"); plt.show()

by_sensor = global_importance.groupby(lambda n: n.split("__")[0]).sum().sort_values(ascending=False)
print("Importance aggregated per anonymized base column (top 8):"); display(by_sensor.head(8).round(4).to_frame("importance"))
global_importance.to_frame("importance").to_csv(ARTIFACT_DIR / "explanations" / "global_feature_importance.csv")

In [ ]:
# Local explanation for one unseen internal-test machine: an early-life row and a late-life row
_u = test_units[0]
_rows = test_df[test_df.unit_id == _u]
_p_u = pd.Series(p_test_cal, index=test_df.index).loc[_rows.index]
_late_idx = _p_u.idxmax(); _early_idx = _rows.index[min(15, len(_rows) - 1)]
local_examples = {}
for label, idx in [("early-life row", _early_idx), ("highest-risk row", _late_idx)]:
    exp = explain_row(final_model, X_test.loc[[idx]], FEATURE_NAMES, reference_values, TOP_K_FEATURES)
    local_examples[label] = {"unit_id": int(_u), "cycle": int(test_df.loc[idx, "cycle"]), "rul_true": int(test_df.loc[idx, "rul"]),
                             "failure_probability": float(_p_u.loc[idx]), **exp}
    print(f"\n--- {label}: unit {_u}, cycle {test_df.loc[idx,'cycle']} | calibrated P(failure within {FAILURE_HORIZON_H}) = {_p_u.loc[idx]:.3f} ---")
    display(pd.DataFrame(exp["top_features"]).round(4))
(ARTIFACT_DIR / "explanations" / "example_local_explanations.json").write_text(json.dumps(to_jsonable(local_examples), indent=2))

## 16. Data Quality / Reliability Checks

Before predicting, the backend runs `check_data_quality`. It returns a status **separate from the ML prediction**:

* `DATA_OK` – nothing unusual found.
* `DATA_WARNING` – prediction is produced but may be less reliable (e.g., short history, a few missing values, some out-of-range values, unexpected extra columns, unsorted cycles or gaps).
* `DATA_INVALID` – prediction is **not** produced (missing required columns, non-numeric values, duplicate/invalid cycles, several units in one request, too many missing values, or a large share of features far outside the training range = unsupported / out-of-distribution input).

A low-quality input must never be reported as a machine failure. Out-of-distribution detection here is a simple, practical range check (training min/max widened by `range_margin_fraction`); it is deliberately different from the Isolation Forest, which scores *machine behaviour* rather than *input validity*.

In [ ]:
QUALITY_SRC = r'''
import numpy as np
import pandas as pd

STATUS_OK, STATUS_WARNING, STATUS_INVALID = "DATA_OK", "DATA_WARNING", "DATA_INVALID"


def _issue(code, severity, message):
    return {"code": code, "severity": severity, "message": message}


def check_data_quality(history, meta):
    """Validate a single machine's history. `meta` = metadata['quality'] saved by the training notebook."""
    cfg = meta["quality_config"]
    required = list(meta["required_columns"])
    schema = set(meta["schema_columns"])
    ranges = meta["feature_ranges"]
    issues = []
    info = {"n_history_rows": 0, "latest_cycle": None, "extreme_fraction": None, "missing_fraction_latest": None}

    def result():
        sev = {i["severity"] for i in issues}
        status = STATUS_INVALID if "INVALID" in sev else STATUS_WARNING if "WARNING" in sev else STATUS_OK
        return {"status": status, "issues": issues, **info}

    if not isinstance(history, pd.DataFrame) or history.empty:
        issues.append(_issue("EMPTY_INPUT", "INVALID", "History is empty or not a DataFrame."))
        return result()
    info["n_history_rows"] = int(len(history))

    missing_cols = [c for c in required if c not in history.columns]
    if missing_cols:
        issues.append(_issue("MISSING_COLUMNS", "INVALID", f"Required columns missing: {missing_cols}"))
        return result()
    unexpected = [c for c in history.columns if c not in schema]
    if unexpected:
        issues.append(_issue("UNEXPECTED_COLUMNS", "WARNING", f"Unexpected columns ignored: {unexpected}"))

    df = history[required + (["unit_id"] if "unit_id" in history.columns and "unit_id" not in required else [])].copy()
    for c in required:
        if not pd.api.types.is_numeric_dtype(df[c]):
            coerced = pd.to_numeric(df[c], errors="coerce")
            if (coerced.isna() & df[c].notna()).any():
                issues.append(_issue("NON_NUMERIC", "INVALID", f"Non-numeric values in column '{c}'."))
            df[c] = coerced
    df[required] = df[required].replace([np.inf, -np.inf], np.nan)
    if any(i["severity"] == "INVALID" for i in issues):
        return result()

    if "unit_id" in df.columns and df["unit_id"].nunique(dropna=False) > 1:
        issues.append(_issue("MULTIPLE_UNITS", "INVALID", "History must contain exactly one machine."))
        return result()

    cyc = df["cycle"]
    if cyc.isna().any():
        issues.append(_issue("CYCLE_MISSING", "INVALID", "Missing cycle values."))
    elif (cyc <= 0).any() or (cyc % 1 != 0).any():
        issues.append(_issue("CYCLE_INVALID", "INVALID", "Cycles must be positive integers."))
    elif cyc.duplicated().any():
        issues.append(_issue("DUPLICATE_CYCLES", "INVALID", "Duplicate cycle numbers."))
    else:
        if not cyc.is_monotonic_increasing:
            issues.append(_issue("UNSORTED_CYCLES", "WARNING", "Cycles not sorted; they will be sorted before scoring."))
        if (np.diff(np.sort(cyc.to_numpy())) > 1).any():
            issues.append(_issue("CYCLE_GAPS", "WARNING", "Gaps between cycles; rolling features may be less reliable."))
    if any(i["severity"] == "INVALID" for i in issues):
        return result()

    df = df.sort_values("cycle", kind="mergesort")
    n = len(df)
    info["latest_cycle"] = int(df["cycle"].iloc[-1])
    if n < cfg["min_history_invalid"]:
        issues.append(_issue("INSUFFICIENT_HISTORY", "INVALID", f"Only {n} row(s); at least {cfg['min_history_invalid']} required."))
        return result()
    if n < meta["min_history_warn"]:
        issues.append(_issue("SHORT_HISTORY", "WARNING", f"{n} rows < {meta['min_history_warn']}: rolling/trend features are based on partial windows."))

    feature_cols = [c for c in required if c != "cycle"]
    latest = df.iloc[-1]
    miss_latest = float(latest[feature_cols].isna().mean())
    info["missing_fraction_latest"] = miss_latest
    if miss_latest > cfg["max_missing_fraction_latest"]:
        issues.append(_issue("MISSING_VALUES_LATEST", "INVALID", f"{miss_latest:.0%} of latest-row values missing."))
        return result()
    if miss_latest > 0:
        issues.append(_issue("MISSING_VALUES_LATEST", "WARNING", f"{miss_latest:.0%} of latest-row values missing (imputed with training medians)."))
    elif df[feature_cols].isna().any().any():
        issues.append(_issue("MISSING_VALUES_HISTORY", "WARNING", "Missing values in earlier history rows."))

    m = float(cfg["range_margin_fraction"])
    checked, extreme = 0, []
    for c in feature_cols:
        v = latest[c]
        if pd.isna(v):
            continue
        checked += 1
        lo, hi = ranges[c]["min"], ranges[c]["max"]
        span = hi - lo
        if v < lo - m * span or v > hi + m * span:
            extreme.append(c)
    info["extreme_fraction"] = (len(extreme) / checked) if checked else 0.0
    if extreme:
        sev = "INVALID" if info["extreme_fraction"] >= cfg["extreme_invalid_fraction"] else "WARNING"
        issues.append(_issue("EXTREME_VALUES" if sev == "WARNING" else "OUT_OF_DISTRIBUTION", sev,
                             f"{len(extreme)} of {checked} latest values outside widened training range (e.g. {extreme[:5]})."))
    return result()
'''
pdm_quality = write_module("pdm_quality", QUALITY_SRC)
from pdm_quality import check_data_quality

_ranges_src = train_df[ACTIVE_SETTINGS + ACTIVE_SENSORS]
QUALITY_META = {
    "required_columns": ["cycle"] + ACTIVE_SETTINGS + ACTIVE_SENSORS,
    "schema_columns": COLUMN_NAMES,
    "feature_ranges": {c: {"min": float(_ranges_src[c].min()), "max": float(_ranges_src[c].max())} for c in _ranges_src.columns},
    "min_history_warn": int(feature_engineer.max_lookback - 1),
    "quality_config": QUALITY_CONFIG,
}

In [ ]:
# Demonstrate the checks on an unseen official-test machine (clean + deliberately corrupted variants)
_hist = test_raw[test_raw.unit_id == sorted(test_raw.unit_id.unique())[0]].copy()
cases = {}
cases["clean history"] = _hist
cases["short history (5 rows)"] = _hist.head(5)
_c = _hist.copy(); _c.loc[_c.index[-1], ACTIVE_SENSORS[:2]] = np.nan; cases["2 sensors missing in latest row"] = _c
_c = _hist.copy(); _c.loc[_c.index[-1], ACTIVE_SENSORS[0]] *= 10; cases["one extreme sensor value"] = _c
_c = _hist.copy(); _c.loc[_c.index[-1], ACTIVE_SENSORS] *= 10; cases["all sensors extreme"] = _c
cases["missing sensor column"] = _hist.drop(columns=[ACTIVE_SENSORS[0]])
_c = _hist.copy(); _c["extra_col"] = 1.0; cases["unexpected extra column"] = _c
cases["two machines in one request"] = test_raw[test_raw.unit_id.isin([1, 2])]
_c = _hist.copy(); _c[ACTIVE_SENSORS[0]] = "bad"; cases["non-numeric values"] = _c
cases["empty"] = _hist.iloc[0:0]

rows = []
for name, h in cases.items():
    r = check_data_quality(h, QUALITY_META)
    rows.append({"case": name, "status": r["status"], "issues": "; ".join(i["code"] for i in r["issues"]) or "-"})
display(pd.DataFrame(rows))
assert rows[0]["status"] == "DATA_OK"

## 17. Model Evaluation

Protocol:

* The **decision threshold** is chosen on **validation units only** (using out-of-fold calibrated probabilities) according to the configured objective (`THRESHOLD_OBJECTIVE`, default F-beta with β = 2, i.e. recall weighted above precision). Section 18 analyses it in detail.
* Final metrics are then computed on **unseen internal-test units** and on the **official `test_FD001`** file (a second unseen set).
* Accuracy is reported but **not** used for decisions. With rare positives, accuracy is dominated by the majority class; **recall** (how many upcoming failures we catch) and **PR-AUC** (precision/recall trade-off over all thresholds, baseline = prevalence) are more informative. **Brier score** and **calibration** measure probability quality.

In [ ]:
def select_threshold(y, p, objective=THRESHOLD_OBJECTIVE, beta=THRESHOLD_BETA, min_precision=THRESHOLD_MIN_PRECISION):
    grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
    rows = []
    for t in grid:
        pred = (p >= t).astype(int)
        rows.append({"threshold": t, "precision": precision_score(y, pred, zero_division=0),
                     "recall": recall_score(y, pred, zero_division=0), "fbeta": fbeta_score(y, pred, beta=beta, zero_division=0)})
    tab = pd.DataFrame(rows)
    if objective == "precision_floor":
        ok = tab[tab.precision >= min_precision]
        if len(ok):
            best = ok.sort_values(["recall", "threshold"], ascending=[False, False]).iloc[0]
            return float(best.threshold), f"max recall subject to precision >= {min_precision}"
        print("No threshold meets the precision floor on validation; falling back to F-beta.")
    best = tab.sort_values(["fbeta", "threshold"], ascending=[False, False]).iloc[0]
    return float(best.threshold), f"max F{beta:g} on validation"

SELECTED_THRESHOLD, THRESHOLD_RULE = select_threshold(y_val, p_val_cal_oof)
print(f"Selected decision threshold = {SELECTED_THRESHOLD:.2f}  ({THRESHOLD_RULE})")

def score_features(X, data_quality_status="DATA_OK"):
    """Vectorised scoring of an imputed feature frame (for evaluation/demos; the backend uses predict_machine_state)."""
    p_raw = final_model.predict_proba(X)[:, 1]
    p_cal = calibrator.transform(p_raw)
    a = anomaly_scorer.score(X)
    hi = compute_health_indicator(p_cal, a["anomaly_severity"], data_quality_status, HEALTH_CONFIG)
    return pd.DataFrame({"p_raw": p_raw, "failure_probability": p_cal, "anomaly_raw": a["raw"],
                         "anomaly_score": a["anomaly_score"], "anomaly_flag": a["anomaly_flag"],
                         "anomaly_severity": a["anomaly_severity"], "health_indicator": hi}, index=X.index)

scored_test = score_features(X_test)
scored_official = score_features(X_official)
last_idx = test_official_df.groupby("unit_id")["cycle"].idxmax()
y_last = test_official_df.loc[last_idx, "failure_within_h"].to_numpy()
p_last = scored_official.loc[last_idx, "failure_probability"].to_numpy()

METRICS = {
    "validation_oof_calibrated": evaluate_probs(y_val, p_val_cal_oof, SELECTED_THRESHOLD),
    "internal_test": evaluate_probs(y_test, scored_test["failure_probability"].to_numpy(), SELECTED_THRESHOLD),
    "official_test_all_rows": evaluate_probs(y_official, scored_official["failure_probability"].to_numpy(), SELECTED_THRESHOLD),
    "official_test_last_cycle_per_unit": evaluate_probs(y_last, p_last, SELECTED_THRESHOLD),
}
metrics_table = pd.DataFrame(METRICS).T[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "brier", "ece"]].astype(float)
print(f"All threshold-dependent metrics use threshold = {SELECTED_THRESHOLD:.2f}; prevalence: internal test {y_test.mean():.3f}, official {y_official.mean():.3f}, official last-cycle {y_last.mean():.3f} (n={len(y_last)})")
display(metrics_table.round(4))
print("Note: the 'last cycle' evaluation has only one row per engine, so its estimates are noisy.")

In [ ]:
# Confusion matrices, ROC and PR curves on unseen data
def plot_eval_panels(sets):
    fig, axes = plt.subplots(len(sets), 3, figsize=(14, 4 * len(sets)))
    axes = np.atleast_2d(axes)
    for row, (name, y_, p_) in zip(axes, sets):
        cm = confusion_matrix(y_, (p_ >= SELECTED_THRESHOLD).astype(int), labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=row[0],
                    xticklabels=["pred no failure", "pred failure<=H"], yticklabels=["true no failure", "true failure<=H"])
        row[0].set_title(f"{name}: confusion matrix @ thr={SELECTED_THRESHOLD:.2f}")
        fpr, tpr, _ = roc_curve(y_, p_); row[1].plot(fpr, tpr); row[1].plot([0, 1], [0, 1], "k--", lw=0.7)
        row[1].set_title(f"ROC (AUC={roc_auc_score(y_, p_):.3f})"); row[1].set_xlabel("FPR"); row[1].set_ylabel("TPR")
        pr, rc, _ = precision_recall_curve(y_, p_); row[2].plot(rc, pr); row[2].axhline(y_.mean(), color="k", ls="--", lw=0.7, label="prevalence")
        row[2].set_title(f"Precision-Recall (AP={average_precision_score(y_, p_):.3f})"); row[2].set_xlabel("recall"); row[2].set_ylabel("precision"); row[2].legend()
    plt.tight_layout()

plot_eval_panels([("Internal test", y_test, scored_test["failure_probability"].to_numpy()),
                  ("Official test_FD001", y_official, scored_official["failure_probability"].to_numpy())])
savefig("evaluation_confusion_roc_pr"); plt.show()
print("Calibration curves: see Section 12 (saved as calibration_before_after.png). Feature importance / SHAP: Section 15.")

## 18. Threshold Analysis

No threshold is universally correct: a low threshold catches more upcoming failures (high recall) but raises more false alarms (low precision); a high threshold does the opposite. The right trade-off depends on the cost of a missed failure versus an unnecessary inspection. The threshold stored in the artifacts is the one selected on validation data under the configurable objective (`THRESHOLD_OBJECTIVE`, `THRESHOLD_BETA`, `THRESHOLD_MIN_PRECISION`); changing the config and re-running regenerates it.

In [ ]:
def threshold_table(y, p, grid):
    rows = []
    for t in grid:
        pred = (p >= t).astype(int)
        rows.append({"threshold": t, "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0),
                     "f1": f1_score(y, pred, zero_division=0), "flagged_share": pred.mean()})
    return pd.DataFrame(rows).set_index("threshold")

thr_val = threshold_table(y_val, p_val_cal_oof, THRESHOLD_GRID)
thr_test = threshold_table(y_test, scored_test["failure_probability"].to_numpy(), THRESHOLD_GRID)
print("Validation (out-of-fold calibrated probabilities):"); display(thr_val.round(3))
print("Internal test:"); display(thr_test.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (title, y_, p_) in zip(axes, [("Validation (OOF)", y_val, p_val_cal_oof), ("Internal test", y_test, scored_test["failure_probability"].to_numpy())]):
    t = threshold_table(y_, p_, np.round(np.arange(0.02, 0.99, 0.02), 2))
    ax.plot(t.index, t["precision"], label="precision"); ax.plot(t.index, t["recall"], label="recall")
    ax.axvline(SELECTED_THRESHOLD, color="k", ls="--", label=f"selected={SELECTED_THRESHOLD:.2f}"); ax.set_title(title); ax.set_xlabel("probability threshold"); ax.legend()
plt.tight_layout(); savefig("threshold_analysis"); plt.show()
thr_val.to_csv(ARTIFACT_DIR / "evaluation" / "threshold_table_validation.csv"); thr_test.to_csv(ARTIFACT_DIR / "evaluation" / "threshold_table_internal_test.csv")

## 19. Machine-Level Prediction Demo

For three **unseen internal-test machines** (shortest, median and longest life) we replay each trajectory cycle by cycle using the trained pipeline and plot: anomaly score → failure probability → Machine Health Indicator, with the **alert** (probability ≥ threshold) and the true failure cycle marked. Everything is computed by the trained models; nothing is hand-written.

In [ ]:
demo_frame = test_df[["unit_id", "cycle", "rul"]].join(scored_test)
life = demo_frame.groupby("unit_id")["cycle"].max().sort_values()
DEMO_UNITS = [life.index[0], life.index[len(life) // 2], life.index[-1]]

def first_alert_cycle(g):
    hit = g[g["failure_probability"] >= SELECTED_THRESHOLD]
    return int(hit["cycle"].iloc[0]) if len(hit) else None

def plot_machine(unit):
    g = demo_frame[demo_frame.unit_id == unit]; fail_cycle = int(g["cycle"].max()); fa = first_alert_cycle(g)
    fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    axes[0].plot(g["cycle"], g["anomaly_score"], color="darkorange"); axes[0].scatter(g.loc[g.anomaly_flag, "cycle"], g.loc[g.anomaly_flag, "anomaly_score"], s=8, color="red", label="anomaly_flag")
    axes[0].set_ylabel("anomaly_score\n(percentile vs healthy)"); axes[0].legend(loc="upper left")
    axes[1].plot(g["cycle"], g["failure_probability"], color="crimson"); axes[1].axhline(SELECTED_THRESHOLD, color="k", ls="--", lw=0.8, label=f"threshold {SELECTED_THRESHOLD:.2f}")
    axes[1].set_ylabel(f"P(failure within {FAILURE_HORIZON_H})"); axes[1].legend(loc="upper left")
    axes[2].plot(g["cycle"], g["health_indicator"], color="seagreen"); axes[2].set_ylabel("Machine Health\nIndicator (0-100)"); axes[2].set_ylim(0, 102); axes[2].set_xlabel("cycle")
    for ax in axes:
        ax.axvline(fail_cycle, color="black", lw=1.2); ax.axvspan(fail_cycle - FAILURE_HORIZON_H, fail_cycle, color="grey", alpha=0.12)
        if fa is not None: ax.axvline(fa, color="purple", ls=":", lw=1.5)
    axes[0].set_title(f"Unit {unit} (unseen): failure at cycle {fail_cycle} (black), last {FAILURE_HORIZON_H} cycles shaded, first alert at cycle {fa} (purple dotted)")
    plt.tight_layout(); savefig(f"demo_unit_{unit}"); plt.show()

for u in DEMO_UNITS:
    plot_machine(u)

In [ ]:
# Alert lead-time summary across ALL internal-test machines (actual measured behaviour)
rows = []
for u, g in demo_frame.groupby("unit_id"):
    fa = first_alert_cycle(g); fail_cycle = int(g["cycle"].max())
    first_flag = g.loc[g.anomaly_flag, "cycle"]
    rows.append({"unit_id": u, "failure_cycle": fail_cycle, "first_anomaly_flag_cycle": int(first_flag.iloc[0]) if len(first_flag) else None,
                 "first_alert_cycle": fa, "cycles_before_failure_at_first_alert": (fail_cycle - fa) if fa is not None else None})
lead = pd.DataFrame(rows).set_index("unit_id")
display(lead)
alerted = lead["cycles_before_failure_at_first_alert"].dropna()
print(f"Units with an alert: {len(alerted)}/{len(lead)} | median cycles before failure at first alert: {alerted.median() if len(alerted) else 'n/a'} "
      f"| alerts raised earlier than H={FAILURE_HORIZON_H} cycles before failure: {(alerted > FAILURE_HORIZON_H).sum()}")

## 20. Maintenance Recommendation Logic

A **transparent, rule-based** layer (no LLM) maps model outputs to one of four categories. It is labelled **"AI recommendation — not a confirmed diagnosis"** (wording configurable via `RECOMMENDATION_CONFIG["label"]`) — it is decision support derived from model outputs, **not** a confirmed diagnosis or a maintenance instruction.

Rules (first match wins; all numbers are configurable in `RECOMMENDATION_CONFIG`):

| Category | Condition |
|---|---|
| `DATA CHECK REQUIRED` | data quality = `DATA_INVALID` (no prediction is made; this is **not** a failure claim) |
| `URGENT REVIEW` | P_fail ≥ max(urgent_probability, threshold) **or** HI ≤ urgent_health_max |
| `SCHEDULE MAINTENANCE` | P_fail ≥ selected threshold **or** HI ≤ schedule_health_max |
| `INSPECT` | anomaly_flag **or** P_fail ≥ 0.5 × threshold **or** HI ≤ inspect_health_max |
| `MONITOR` | otherwise |

In [ ]:
RECOMMEND_SRC = r'''
DEFAULT_RECOMMENDATION_CONFIG = {
    "urgent_probability": 0.80,
    "urgent_health_max": 25.0,
    "schedule_health_max": 50.0,
    "inspect_health_max": 75.0,
    "inspect_probability_fraction_of_threshold": 0.5,
}
CATEGORIES = ("MONITOR", "INSPECT", "SCHEDULE MAINTENANCE", "URGENT REVIEW")
LABEL = "AI recommendation - not a confirmed diagnosis"
DISCLAIMER = ("This is an AI-generated recommendation derived from model outputs by transparent rules. It is not a "
              "confirmed diagnosis or a maintenance instruction; a qualified person should make the final decision.")


def recommend_maintenance(failure_probability, anomaly_flag, health_indicator, data_quality_status,
                          decision_threshold, config=None):
    cfg = {**DEFAULT_RECOMMENDATION_CONFIG, **(config or {})}
    reasons = []
    if data_quality_status == "DATA_INVALID":
        return {"label": cfg.get("label", LABEL), "category": "DATA CHECK REQUIRED",
                "reasons": ["Input data failed validation; no prediction was made. This is not a failure claim."],
                "disclaimer": cfg.get("disclaimer", DISCLAIMER)}
    p, hi, thr = float(failure_probability), float(health_indicator), float(decision_threshold)
    urgent_p = max(cfg["urgent_probability"], thr)
    if p >= urgent_p or hi <= cfg["urgent_health_max"]:
        category = "URGENT REVIEW"
        reasons.append(f"Failure probability {p:.2f} >= {urgent_p:.2f}" if p >= urgent_p else f"Health indicator {hi:.1f} <= {cfg['urgent_health_max']}")
    elif p >= thr or hi <= cfg["schedule_health_max"]:
        category = "SCHEDULE MAINTENANCE"
        reasons.append(f"Failure probability {p:.2f} >= decision threshold {thr:.2f}" if p >= thr else f"Health indicator {hi:.1f} <= {cfg['schedule_health_max']}")
    elif anomaly_flag or p >= cfg["inspect_probability_fraction_of_threshold"] * thr or hi <= cfg["inspect_health_max"]:
        category = "INSPECT"
        if anomaly_flag:
            reasons.append("Anomaly detector flags unusual behaviour relative to healthy training data")
        if p >= cfg["inspect_probability_fraction_of_threshold"] * thr:
            reasons.append(f"Failure probability {p:.2f} is approaching the decision threshold {thr:.2f}")
        if hi <= cfg["inspect_health_max"]:
            reasons.append(f"Health indicator {hi:.1f} <= {cfg['inspect_health_max']}")
    else:
        category = "MONITOR"
        reasons.append("No rule triggered: low failure probability, no anomaly flag, health indicator high")
    if data_quality_status == "DATA_WARNING":
        reasons.append("Data-quality warning present: verify inputs before acting")
    return {"label": cfg.get("label", LABEL), "category": category, "reasons": reasons, "disclaimer": cfg.get("disclaimer", DISCLAIMER)}
'''
pdm_recommendation = write_module("pdm_recommendation", RECOMMEND_SRC)
from pdm_recommendation import recommend_maintenance

def recommend_frame(frame):
    return [recommend_maintenance(r.failure_probability, r.anomaly_flag, r.health_indicator, "DATA_OK", SELECTED_THRESHOLD, RECOMMENDATION_CONFIG)["category"]
            for r in frame.itertuples()]

demo_frame["recommendation"] = recommend_frame(demo_frame)
order = ["MONITOR", "INSPECT", "SCHEDULE MAINTENANCE", "URGENT REVIEW"]
print("First cycle at which each category appears for the demo machines (and the machine's failure cycle):")
tl = []
for u in DEMO_UNITS:
    g = demo_frame[demo_frame.unit_id == u]
    tl.append({"unit_id": u, "failure_cycle": int(g.cycle.max()), **{c: (int(g.loc[g.recommendation == c, "cycle"].min()) if (g.recommendation == c).any() else None) for c in order}})
display(pd.DataFrame(tl).set_index("unit_id"))

demo_frame["rul_bucket"] = pd.cut(demo_frame["rul"], [-1, FAILURE_HORIZON_H, HEALTHY_MIN_RUL - 1, 10_000], labels=[f"RUL<={FAILURE_HORIZON_H}", f"{FAILURE_HORIZON_H}<RUL<{HEALTHY_MIN_RUL}", f"RUL>={HEALTHY_MIN_RUL}"])
print("Share of recommendation categories by true remaining life (all internal-test rows):")
display(pd.crosstab(demo_frame["rul_bucket"], demo_frame["recommendation"], normalize="index").reindex(columns=order, fill_value=0).round(3))

## 21. End-to-End Inference Function

**21.1 Metadata (the contract with the backend).** All thresholds/configs/feature lists the backend needs are assembled into one JSON-serialisable dictionary. Only *measured* metrics are written.

In [ ]:
library_versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                    "scikit-learn": sklearn.__version__, "scipy": scipy.__version__, "joblib": joblib.__version__,
                    "xgboost": xgb.__version__ if XGB_AVAILABLE else None, "shap": shap.__version__ if SHAP_AVAILABLE else None}

METRICS["anomaly_detection_proxy"] = anomaly_eval.to_dict(orient="index")
METRICS["candidate_models_validation_raw"] = baseline_table[["roc_auc", "pr_auc", "brier", "ece"]].astype(float).to_dict(orient="index")
METRICS["calibration_comparison_validation_oof"] = calib_compare.to_dict(orient="index")
METRICS["calibration_before_after"] = before_after.to_dict(orient="index")
METRICS["anomaly_flag_rates_healthy_rows"] = ANOMALY_FLAG_RATES

METADATA = to_jsonable({
    "model_version": MODEL_VERSION,
    "dataset": f"NASA C-MAPSS {DATASET_ID}",
    "dataset_info": {
        "description": "Simulated turbofan-engine run-to-failure benchmark (NASA C-MAPSS). Sensor columns are ANONYMIZED; no physical meaning is assigned.",
        "is_simulated": True, "sensor_naming": "anonymized sensor_1..sensor_21",
        "train_file_units": int(train_raw.unit_id.nunique()), "train_file_rows": int(len(train_raw)),
        "official_test_units": int(test_raw.unit_id.nunique()), "official_test_rows": int(len(test_raw)),
        "units_train_split": len(train_units), "units_validation_split": len(val_units), "units_internal_test_split": len(test_units),
        "split_unit_ids": {"train": train_units, "validation": val_units, "internal_test": test_units},
        "split_policy": "by unit_id (no machine appears in more than one split)"},
    "target": "failure_within_H_cycles",
    "target_definition": "1 if remaining useful life (cycles) <= failure_horizon else 0",
    "failure_horizon": FAILURE_HORIZON_H, "failure_horizon_unit": "operating_cycles",
    "anomaly_model": "IsolationForest",
    "failure_model": PRIMARY_NAME, "failure_model_class": type(final_model).__name__,
    "calibration_method": CALIBRATION_METHOD,
    "class_weighting_used": bool(USE_CLASS_WEIGHTS), "train_positive_rate": TRAIN_POSITIVE_RATE,
    "features": FEATURE_NAMES,
    "feature_config": {"active_sensors": ACTIVE_SENSORS, "active_operating_settings": ACTIVE_SETTINGS,
                       "dropped_sensors_low_variance": DROPPED_SENSORS, "dropped_operating_settings": DROPPED_SETTINGS,
                       "rolling_windows": list(ROLLING_WINDOWS), "slope_window": SLOPE_WINDOW,
                       "pct_change_eps": PCT_CHANGE_EPS, "pct_change_clip": PCT_CHANGE_CLIP,
                       "imputer": "median (fitted on training units)", "anomaly_features": ANOMALY_FEATURES,
                       "max_lookback_rows": feature_engineer.max_lookback},
    "schema": {"required_input_columns": QUALITY_META["required_columns"], "optional_input_columns": ["unit_id"],
               "full_dataset_schema": COLUMN_NAMES, "one_row_per_operating_cycle": True,
               "note": "Send the machine's chronological history (at least max_lookback rows recommended)."},
    "threshold": {"value": SELECTED_THRESHOLD, "objective": THRESHOLD_OBJECTIVE, "beta": THRESHOLD_BETA,
                  "min_precision": THRESHOLD_MIN_PRECISION, "rule": THRESHOLD_RULE,
                  "selected_on": "validation units, out-of-fold calibrated probabilities"},
    "anomaly": {"contamination": ANOMALY_CONTAMINATION, "healthy_min_rul": HEALTHY_MIN_RUL,
                "alarm_threshold_calibrated_on": "healthy rows of validation units (not seen by the forest)",
                "flag_rates_healthy_rows": ANOMALY_FLAG_RATES,
                "score_semantics": "raw = -score_samples (higher = more anomalous); anomaly_score = percentile vs healthy training raw scores; not a probability"},
    "health_indicator": {"name": "Machine Health Indicator", "trained_model": False,
                         "formula": "clip(100*(1-P_fail)*(1-anomaly_weight*anomaly_severity) - data_quality_penalty, 0, 100)",
                         "additive_decomposition": {"failure_risk_points": "100*P_fail", "anomaly_points": "100*(1-P_fail)*anomaly_weight*anomaly_severity",
                                                    "data_quality_points": "data_quality_penalty", "trend_points": "not enabled"},
                         "config": HEALTH_CONFIG},
    "recommendation_rules": {"label": RECOMMENDATION_CONFIG["label"], "config": RECOMMENDATION_CONFIG},
    "quality": QUALITY_META,
    "failure_horizon_status": "configured_not_optimized", "failure_horizon_rationale": FAILURE_HORIZON_RATIONALE,
    "reliability_mapping": {"DATA_OK": "reliable", "DATA_WARNING": "reliability_reduced", "DATA_INVALID": "not_scored"},
    "training_timestamp": TRAINING_TIMESTAMP,
    "random_state": RANDOM_STATE,
    "library_versions": library_versions,
    "metrics": METRICS,
})
print(f"Metadata assembled: {len(METADATA)} top-level keys")

**21.2 The inference module.** `predict_machine_state(history, bundle, ...)` is the single entry point the backend will call. It lives in `artifacts/code/pdm_inference.py`, contains **no training logic**, and uses only saved artifacts. Order of operations: data-quality check → (if not invalid) causal features → impute → calibrated failure probability → anomaly scores → health indicator → status → explanation → recommendation. Data quality is reported separately from the ML prediction, and `DATA_INVALID` inputs are **not scored**.

In [ ]:
INFERENCE_SRC = r'''
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from pdm_quality import check_data_quality
from pdm_health import compute_health_indicator, health_breakdown
from pdm_recommendation import recommend_maintenance
from pdm_explain import explain_row

# Data-quality status -> user-facing reliability wording
RELIABILITY = {"DATA_OK": "reliable", "DATA_WARNING": "reliability_reduced", "DATA_INVALID": "not_scored"}


class DataInvalidError(ValueError):
    """Raised by score_trajectory when the input fails validation (the result holds the quality report)."""

    def __init__(self, quality):
        super().__init__("; ".join(i["message"] for i in quality["issues"]) or "invalid input")
        self.quality = quality


class ModelBundle:
    """Everything needed for inference. Load with ModelBundle.load(artifact_dir).

    SECURITY: pickled classes come from artifact_dir/code. Verify artifact_manifest.json
    (verify_artifacts.verify_all) BEFORE putting that folder on sys.path / calling load().
    """

    def __init__(self, feature_engineer, imputer, calibrated_model, anomaly_scorer, metadata):
        self.feature_engineer = feature_engineer
        self.imputer = imputer
        self.calibrated_model = calibrated_model
        self.anomaly_scorer = anomaly_scorer
        self.metadata = metadata
        self.feature_names = list(metadata["features"])
        self.reference_values = pd.Series(imputer.statistics_, index=self.feature_names)

    @classmethod
    def load(cls, artifact_dir):
        d = Path(artifact_dir)
        code_dir = str(d / "code")
        if code_dir not in sys.path:
            sys.path.insert(0, code_dir)
        metadata = json.loads((d / "metadata" / "model_metadata.json").read_text())
        return cls(joblib.load(d / "preprocessing" / "feature_engineer.joblib"),
                   joblib.load(d / "preprocessing" / "imputer.joblib"),
                   joblib.load(d / "model" / "calibrated_failure_model.joblib"),
                   joblib.load(d / "model" / "anomaly_detector.joblib"),
                   metadata)


def _py(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    return v


def _prepare_history(history, bundle, lookback=True):
    req = list(bundle.metadata["quality"]["required_columns"])
    cols = req + (["unit_id"] if "unit_id" in history.columns and "unit_id" not in req else [])
    df = history[cols].copy()
    for c in req:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[req] = df[req].replace([np.inf, -np.inf], np.nan)
    df = df.sort_values("cycle", kind="mergesort")
    if lookback:
        df = df.tail(bundle.feature_engineer.max_lookback)
    return df.reset_index(drop=True)


def _impute(bundle, feats):
    return pd.DataFrame(bundle.imputer.transform(feats), columns=bundle.feature_names, index=feats.index)


def predict_machine_state(history, bundle, machine_id=None, top_k=5, include_explanation=True):
    """Score the LATEST cycle of one machine's chronological history. Returns a JSON-serialisable dict."""
    meta = bundle.metadata
    threshold = float(meta["threshold"]["value"])
    horizon = int(meta["failure_horizon"])
    rec_cfg = meta["recommendation_rules"]["config"]
    dq = check_data_quality(history, meta["quality"])
    if machine_id is None and isinstance(history, pd.DataFrame) and "unit_id" in history.columns and len(history):
        machine_id = _py(history["unit_id"].iloc[0])
    base = {"machine_id": _py(machine_id), "cycle": dq.get("latest_cycle"), "failure_horizon": horizon,
            "data_quality_status": dq["status"], "reliability": RELIABILITY[dq["status"]],
            "data_quality_issues": dq["issues"], "model_version": meta["model_version"], "decision_threshold": threshold}

    if dq["status"] == "DATA_INVALID":
        rec = recommend_maintenance(0.0, False, 0.0, "DATA_INVALID", threshold, rec_cfg)
        return {**base, "failure_probability": None, "anomaly_score": None, "anomaly_raw_score": None, "anomaly_flag": None,
                "machine_health_indicator": None, "health_breakdown": None, "prediction_status": "NOT_EVALUATED",
                "top_contributing_features": [], "maintenance_recommendation": rec}

    frame = _prepare_history(history, bundle, lookback=True)
    X = _impute(bundle, bundle.feature_engineer.transform(frame).iloc[[-1]])

    p = float(bundle.calibrated_model.predict_proba(X)[0, 1])
    a = bundle.anomaly_scorer.score(X)
    severity = float(a["anomaly_severity"][0])
    flag = bool(a["anomaly_flag"][0])
    hi_cfg = meta["health_indicator"]["config"]
    hi = compute_health_indicator(p, severity, dq["status"], hi_cfg)
    breakdown = health_breakdown(p, severity, dq["status"], hi_cfg)
    rec = recommend_maintenance(p, flag, hi, dq["status"], threshold, rec_cfg)

    top, method = [], None
    if include_explanation:
        try:
            exp = explain_row(bundle.calibrated_model.base_model, X, bundle.feature_names, bundle.reference_values, top_k)
            top, method = exp["top_features"], exp["method"]
        except Exception as exc:  # explanations must never break a prediction
            method = f"unavailable: {exc}"

    return {**base,
            "failure_probability": round(p, 6),
            "anomaly_score": round(float(a["anomaly_score"][0]), 6),
            "anomaly_raw_score": round(float(a["raw"][0]), 6),
            "anomaly_flag": flag,
            "machine_health_indicator": hi,
            "health_breakdown": breakdown,
            "prediction_status": "ALERT" if p >= threshold else "NO_ALERT",
            "top_contributing_features": top,
            "explanation_method": method,
            "maintenance_recommendation": rec}


def score_trajectory(history, bundle, include_recommendation=True):
    """Vectorised per-cycle scoring of a whole chronological history (same maths as predict_machine_state).

    Data quality is evaluated once for the submitted history and applied to every row. Raises DataInvalidError
    for invalid input. Use this (not one predict_machine_state call per cycle) for trajectories / seeding.
    """
    meta = bundle.metadata
    threshold = float(meta["threshold"]["value"])
    dq = check_data_quality(history, meta["quality"])
    if dq["status"] == "DATA_INVALID":
        raise DataInvalidError(dq)
    frame = _prepare_history(history, bundle, lookback=False)
    X = _impute(bundle, bundle.feature_engineer.transform(frame))
    p = bundle.calibrated_model.predict_proba(X)[:, 1]
    a = bundle.anomaly_scorer.score(X)
    hi = compute_health_indicator(p, a["anomaly_severity"], dq["status"], meta["health_indicator"]["config"])
    out = pd.DataFrame({"cycle": frame["cycle"].astype(int).to_numpy(),
                        "failure_probability": p, "anomaly_raw_score": a["raw"], "anomaly_score": a["anomaly_score"],
                        "anomaly_flag": a["anomaly_flag"], "anomaly_severity": a["anomaly_severity"],
                        "machine_health_indicator": hi,
                        "prediction_status": np.where(p >= threshold, "ALERT", "NO_ALERT")})
    if "unit_id" in frame.columns:
        out.insert(0, "machine_id", frame["unit_id"].to_numpy())
    if include_recommendation:
        out["recommendation_category"] = [
            recommend_maintenance(pp, ff, hh, dq["status"], threshold, meta["recommendation_rules"]["config"])["category"]
            for pp, ff, hh in zip(out["failure_probability"], out["anomaly_flag"], out["machine_health_indicator"])]
    out["data_quality_status"] = dq["status"]
    out["reliability"] = RELIABILITY[dq["status"]]
    out.attrs["data_quality"] = dq
    return out
'''
pdm_inference = write_module("pdm_inference", INFERENCE_SRC)
from pdm_inference import ModelBundle, predict_machine_state

**21.3 Test the inference function (in-memory bundle).** We (a) score three unseen official-test machines, (b) verify the output is JSON-serialisable, and (c) verify **consistency**: scoring a truncated history through `predict_machine_state` must reproduce the vectorised evaluation pipeline exactly.

In [ ]:
bundle_mem = ModelBundle(feature_engineer, imputer, calibrated_model, anomaly_scorer, METADATA)

sample_results = []
for u in sorted(test_raw.unit_id.unique())[:3]:
    res = predict_machine_state(test_raw[test_raw.unit_id == u], bundle_mem)
    json.dumps(res)                                          # must be JSON-serialisable
    sample_results.append(res)
print(json.dumps(sample_results[0], indent=2)[:2500])
display(pd.DataFrame([{k: r[k] for k in ["machine_id", "cycle", "failure_probability", "anomaly_score", "anomaly_flag", "machine_health_indicator",
                                          "prediction_status", "data_quality_status"]} | {"recommendation": r["maintenance_recommendation"]["category"]} for r in sample_results]))

# consistency: per-cycle API result == vectorised evaluation result
_u = DEMO_UNITS[1]; _g = test_df[test_df.unit_id == _u]
for k in [30, 80, int(_g.cycle.max())]:
    hist = _g[_g.cycle <= k][COLUMN_NAMES]
    r = predict_machine_state(hist, bundle_mem)
    vec = scored_test.loc[_g[_g.cycle == k].index[0]]
    assert r["data_quality_status"] == "DATA_OK", r["data_quality_issues"]
    assert abs(r["failure_probability"] - vec["failure_probability"]) < 1e-5 and abs(r["anomaly_raw_score"] - vec["anomaly_raw"]) < 1e-5
print("Consistency check passed: predict_machine_state == vectorised pipeline.")

# vectorised trajectory scoring (what the backend should use for whole histories)
from pdm_inference import score_trajectory
_traj = score_trajectory(_g[COLUMN_NAMES], bundle_mem)
_vec = scored_test.loc[_g.index]
assert np.allclose(_traj["failure_probability"].to_numpy(), _vec["failure_probability"].to_numpy(), atol=1e-5)
assert np.allclose(_traj["machine_health_indicator"].to_numpy(), _vec["health_indicator"].to_numpy(), atol=0.011)
assert (_traj["anomaly_flag"].to_numpy() == _vec["anomaly_flag"].to_numpy()).all()
print("score_trajectory == vectorised evaluation pipeline == predict_machine_state at sampled cycles.")
display(_traj.tail(3))
print("Health breakdown (latest cycle of the last check):", r["health_breakdown"])

bad = predict_machine_state(test_raw[test_raw.unit_id == 1].drop(columns=[ACTIVE_SENSORS[0]]), bundle_mem)
print("\nInvalid input ->", bad["data_quality_status"], "|", bad["prediction_status"], "|", bad["maintenance_recommendation"]["category"], "| probability:", bad["failure_probability"])

## 22. Production Artifact Generation

Directory layout written for the backend (the notebook is the **source of truth** for the bundle layout and for all ML code):

```
artifacts/
  model/          failure_model_raw.joblib, calibrated_failure_model.joblib, probability_calibrator.joblib, anomaly_detector.joblib, zscore_baseline.joblib
  preprocessing/  feature_engineer.joblib, imputer.joblib
  evaluation/     metrics.json, curves.json (ROC / PR / confusion counts / calibration bins / threshold tables), feature_importance.json,
                  figures (*.png), CSV tables
  metadata/       model_metadata.json (single source of truth), model_card.json, MODEL_CARD.md, feature_config.json, health_indicator_config.json,
                  schema.json, dataset_info.json, artifact_manifest.json (SHA-256 of every file), final_summary.txt
  explanations/   global_feature_importance.csv, shap_summary.png, example_local_explanations.json
  demo/           demo_units.csv (raw FD001-format rows), demo_units_ground_truth.csv, demo_reference_scores.csv, demo_manifest.json
  code/           pdm_*.py (feature engineering, calibration, anomaly, health, quality, explain, recommendation, inference) - vendor these in the backend
  verify_artifacts.py   stdlib-only verifier (manifest SHA-256 + library-version check) - vendor this too
  requirements-inference.txt   exact library versions the artifacts were produced with
```

`joblib` is used for Python objects; JSON for everything a human or service reads. **Security / compatibility:** the pickled classes come from `code/`, so the backend must verify every file against `artifact_manifest.json` **before** adding `code/` to `sys.path` and refuse to load if installed library versions differ from `metadata.library_versions` (both done by `verify_artifacts.verify_all`).

In [ ]:
A = ARTIFACT_DIR
joblib.dump(final_model, A / "model" / "failure_model_raw.joblib")
joblib.dump(calibrated_model, A / "model" / "calibrated_failure_model.joblib")
joblib.dump(calibrator, A / "model" / "probability_calibrator.joblib")
joblib.dump(anomaly_scorer, A / "model" / "anomaly_detector.joblib")
joblib.dump(zscore_baseline, A / "model" / "zscore_baseline.joblib")
joblib.dump(feature_engineer, A / "preprocessing" / "feature_engineer.joblib")
joblib.dump(imputer, A / "preprocessing" / "imputer.joblib")

def dump_json(obj, path):
    Path(path).write_text(json.dumps(to_jsonable(obj), indent=2))

dump_json(METADATA, A / "metadata" / "model_metadata.json")
dump_json(METADATA["feature_config"] | {"features": FEATURE_NAMES}, A / "metadata" / "feature_config.json")
dump_json(METADATA["health_indicator"], A / "metadata" / "health_indicator_config.json")
dump_json(METADATA["schema"] | {"feature_ranges": QUALITY_META["feature_ranges"]}, A / "metadata" / "schema.json")
dump_json(METADATA["dataset_info"], A / "metadata" / "dataset_info.json")
dump_json(METRICS, A / "evaluation" / "metrics.json")
metrics_table.to_csv(A / "evaluation" / "metrics_summary.csv")
calib_compare.to_csv(A / "evaluation" / "calibration_method_comparison.csv")
baseline_table.to_csv(A / "evaluation" / "candidate_models_validation.csv")
lead.to_csv(A / "evaluation" / "alert_lead_times_internal_test.csv")
print("Core artifacts saved.")

### 22.1 Structured evaluation data for the Model Performance page

PNG images are not enough for a UI, so the **curve points, confusion-matrix counts, calibration bins, threshold tables and feature importances** are exported as data. Curves are thinned to at most `CURVE_MAX_POINTS` points. All values are computed from the same probabilities used for the reported metrics (calibrated probabilities; calibration bins show raw vs calibrated).

In [ ]:
import re

def _thin(n, max_points):
    return np.unique(np.linspace(0, n - 1, min(n, max_points)).astype(int))

def calibration_bins(y, p, n_bins=CALIBRATION_BINS):
    s = pd.DataFrame({"y": np.asarray(y).astype(int), "p": np.asarray(p, float)})
    s["bin"] = pd.qcut(s["p"].rank(method="first"), int(min(n_bins, len(s))), labels=False)
    g = s.groupby("bin").agg(mean_predicted=("p", "mean"), observed_frequency=("y", "mean"), count=("y", "size"))
    return g.reset_index(drop=True).to_dict("records")

def curve_payload(y, p_cal, threshold, metrics, p_raw=None):
    y = np.asarray(y).astype(int); p_cal = np.asarray(p_cal, float)
    fpr, tpr, thr_roc = roc_curve(y, p_cal); i = _thin(len(fpr), CURVE_MAX_POINTS)
    prec, rec, thr_pr = precision_recall_curve(y, p_cal); thr_pr = np.append(thr_pr, 1.0); j = _thin(len(prec), CURVE_MAX_POINTS)
    tn, fp, fn, tp = confusion_matrix(y, (p_cal >= threshold).astype(int), labels=[0, 1]).ravel()
    out = {"n_rows": int(len(y)), "n_positive": int(y.sum()), "prevalence": float(y.mean()), "threshold": float(threshold),
           "metrics_at_threshold": metrics,
           "confusion_matrix": {"labels": ["no_failure_within_H", "failure_within_H"], "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
                                "matrix_rows_true_cols_pred": [[int(tn), int(fp)], [int(fn), int(tp)]]},
           "roc": {"fpr": fpr[i], "tpr": tpr[i], "thresholds": np.clip(thr_roc[i], 0, 1), "auc": roc_auc_score(y, p_cal)},
           "precision_recall": {"precision": prec[j], "recall": rec[j], "thresholds": thr_pr[j],
                                "average_precision": average_precision_score(y, p_cal), "prevalence_baseline": float(y.mean())}}
    if p_raw is not None:
        out["calibration"] = {"method": CALIBRATION_METHOD, "n_bins": CALIBRATION_BINS, "binning": "equal-count (quantile) bins",
                              "bins_raw": calibration_bins(y, p_raw), "bins_calibrated": calibration_bins(y, p_cal),
                              "brier_raw": brier_score_loss(y, p_raw), "brier_calibrated": brier_score_loss(y, p_cal),
                              "ece_raw": ece_score(y, p_raw), "ece_calibrated": ece_score(y, p_cal)}
    return out

_pt = scored_test["failure_probability"].to_numpy(); _po = scored_official["failure_probability"].to_numpy()
CURVES = {
    "meta": {"model_version": MODEL_VERSION, "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
             "probabilities": "calibrated P(failure within H cycles)", "failure_horizon": FAILURE_HORIZON_H,
             "selected_threshold": SELECTED_THRESHOLD, "max_points_per_curve": CURVE_MAX_POINTS},
    "validation_oof": curve_payload(y_val, p_val_cal_oof, SELECTED_THRESHOLD, METRICS["validation_oof_calibrated"], p_raw=raw_val),
    "internal_test": curve_payload(y_test, _pt, SELECTED_THRESHOLD, METRICS["internal_test"], p_raw=raw_test),
    "official_test_all_rows": curve_payload(y_official, _po, SELECTED_THRESHOLD, METRICS["official_test_all_rows"], p_raw=p_off_raw),
    "official_test_last_cycle_per_unit": curve_payload(y_last, p_last, SELECTED_THRESHOLD, METRICS["official_test_last_cycle_per_unit"]),
    "threshold_analysis": {"grid": THRESHOLD_GRID, "validation_oof": thr_val.reset_index().to_dict("records"),
                           "internal_test": thr_test.reset_index().to_dict("records")},
    "alert_lead_times_internal_test": lead.reset_index().to_dict("records"),
}
dump_json(CURVES, A / "evaluation" / "curves.json")

def feature_group(name):
    if name == "cycle": return "cycle"
    if name.startswith("operating_setting"): return "operating_setting"
    if "__" not in name: return "raw_sensor"
    return re.sub(r"_\d+$", "", name.split("__", 1)[1])

_gi = global_importance.clip(lower=0)
FEATURE_IMPORTANCE = {
    "model_version": MODEL_VERSION, "method": GLOBAL_IMPORTANCE_METHOD,
    "scale": "mean |log-odds contribution| of the UNCALIBRATED model" if kind == "xgboost" else "drop in PR-AUC when the feature is permuted (validation units)",
    "n_features": int(len(global_importance)),
    "note": "sensor_k columns are anonymized; names such as sensor_k__slope_10 mean 'recent trend of anonymized sensor k'. No physical meaning is implied.",
    "features": [{"feature": f, "importance": float(v), "importance_share": float(v / _gi.sum()) if _gi.sum() > 0 else None,
                  "base_column": f.split("__")[0], "feature_group": feature_group(f)} for f, v in global_importance.head(40).items()],
    "by_base_column": [{"base_column": b, "importance": float(v)} for b, v in by_sensor.items()],
    "by_feature_group": [{"feature_group": g, "importance": float(v)} for g, v in
                         global_importance.groupby(feature_group).sum().sort_values(ascending=False).items()],
}
dump_json(FEATURE_IMPORTANCE, A / "evaluation" / "feature_importance.json")
print("curves.json sets:", [k for k in CURVES if k not in ("meta", "threshold_analysis", "alert_lead_times_internal_test")])
print("curves.json size:", round((A / "evaluation" / "curves.json").stat().st_size / 1024, 1), "KB | feature_importance.json features:", len(FEATURE_IMPORTANCE["features"]))

### 22.2 Demo data export (for the backend seed)

A handful of **raw FD001-format rows** for engines chosen by a **fixed random seed** (`DEMO_SEED`) from the **internal-test units** (never seen in training/validation). The units are *not* chosen by looking at model outputs. Also exported: ground-truth RUL/label columns (for display only) and reference scores from the vectorised scorer so the backend can run a parity test.

In [ ]:
from pdm_inference import score_trajectory
(A / "demo").mkdir(exist_ok=True)
_demo_rng = np.random.RandomState(DEMO_SEED)
DEMO_EXPORT_UNITS = sorted(int(u) for u in _demo_rng.choice(test_units, size=min(N_DEMO_UNITS, len(test_units)), replace=False))
demo_rows = train_labeled[train_labeled.unit_id.isin(DEMO_EXPORT_UNITS)]
demo_rows[COLUMN_NAMES].to_csv(A / "demo" / "demo_units.csv", index=False)
demo_rows[["unit_id", "cycle", "rul", "failure_within_h"]].to_csv(A / "demo" / "demo_units_ground_truth.csv", index=False)
ref_scores = pd.concat([score_trajectory(g[COLUMN_NAMES], bundle_mem) for _, g in demo_rows.groupby("unit_id")], ignore_index=True)
ref_scores.to_csv(A / "demo" / "demo_reference_scores.csv", index=False)
dump_json({"selection": "fixed-seed random choice among internal-test units (not chosen using model outputs)", "seed": DEMO_SEED,
           "unit_ids": DEMO_EXPORT_UNITS, "source_split": "internal_test", "n_rows": int(len(demo_rows)),
           "format": "demo_units.csv = raw FD001 columns (unit_id, cycle, operating_setting_1..3, sensor_1..21)",
           "reference_scores": "demo_reference_scores.csv = outputs of score_trajectory for parity testing",
           "model_version": MODEL_VERSION, "note": "Rows come from the C-MAPSS-format training file; the data is simulated and sensors are anonymized."},
          A / "demo" / "demo_manifest.json")
print("Demo engines:", DEMO_EXPORT_UNITS, "| rows:", len(demo_rows))

### 22.3 Verifier and version pins

`verify_artifacts.py` is **standard-library only** so it can be vendored into the backend and run *before* any artifact code is imported. `requirements-inference.txt` lists the exact versions the artifacts were produced with (pin the backend to these; do not downgrade Colab to older pins).

In [ ]:
VERIFY_SRC = r'''
"""Standalone, stdlib-only artifact verifier. VENDOR this file into the backend (do not import it from the artifact folder).

Run verify_all(artifact_dir) BEFORE adding artifact_dir/code to sys.path or unpickling anything.
"""
import hashlib
import json
import platform
from importlib import metadata as importlib_metadata
from pathlib import Path


class ArtifactVerificationError(RuntimeError):
    pass


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def verify_manifest(artifact_dir, allow_unlisted=False):
    root = Path(artifact_dir)
    mpath = root / "metadata" / "artifact_manifest.json"
    if not mpath.is_file():
        raise ArtifactVerificationError("artifact_manifest.json not found")
    files = json.loads(mpath.read_text())["files"]
    problems = []
    for rel, info in files.items():
        p = root / rel
        if not p.is_file():
            problems.append(f"missing: {rel}")
        elif p.stat().st_size != info["bytes"] or _sha256(p) != info["sha256"]:
            problems.append(f"hash mismatch: {rel}")
    if not allow_unlisted:
        for p in root.rglob("*"):
            if p.is_file() and "__pycache__" not in p.parts and p.suffix != ".pyc":
                rel = p.relative_to(root).as_posix()
                if rel not in files and rel != "metadata/artifact_manifest.json":
                    problems.append(f"unlisted file: {rel}")
    if problems:
        raise ArtifactVerificationError("; ".join(problems[:10]))
    return True


def check_library_versions(metadata, strict=True):
    """Compare installed versions with metadata['library_versions'] (python: major.minor only)."""
    problems = []
    for name, want in metadata.get("library_versions", {}).items():
        if want is None or name == "shap":      # shap is training/explanation-plot only
            continue
        if name == "python":
            have, want = ".".join(platform.python_version().split(".")[:2]), ".".join(want.split(".")[:2])
        else:
            try:
                have = importlib_metadata.version(name)
            except importlib_metadata.PackageNotFoundError:
                problems.append(f"{name}: required {want}, not installed")
                continue
        if have != want:
            problems.append(f"{name}: artifacts built with {want}, installed {have}")
    if problems and strict:
        raise ArtifactVerificationError("library version mismatch: " + "; ".join(problems))
    return problems


def verify_all(artifact_dir, strict_versions=True):
    verify_manifest(artifact_dir)
    metadata = json.loads((Path(artifact_dir) / "metadata" / "model_metadata.json").read_text())
    check_library_versions(metadata, strict=strict_versions)
    return metadata
'''
(A / "verify_artifacts.py").write_text(textwrap.dedent(VERIFY_SRC).lstrip("\n"))
_req = [f"{k}=={v}" for k, v in library_versions.items() if k not in ("python", "shap") and v]
(A / "requirements-inference.txt").write_text(f"# Python {'.'.join(library_versions['python'].split('.')[:2])} (artifacts built on {library_versions['python']})\n"
                                              "# Exact versions used to create the artifacts; the backend refuses to load on mismatch.\n" + "\n".join(_req) + "\n")
print((A / "requirements-inference.txt").read_text())

## 23. Model Card

Generated from the **actual** metadata and measured metrics. Two forms: `MODEL_CARD.md` (human-readable) and `model_card.json` (structured, for the UI), including the **reason for choosing H** and the **evaluation date**. H is documented as *configured, not optimized*.

In [ ]:
def fmt(x, nd=3):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{nd}f}"

EVALUATION_DATE_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")
mt, mi, mo = METRICS["internal_test"], METRICS["validation_oof_calibrated"], METRICS["official_test_all_rows"]
LIMITATIONS = [
    "Trained and evaluated on a SIMULATED single-operating-condition dataset (FD001); results may not transfer to real machines or other C-MAPSS subsets.",
    "Small number of engines; metrics carry sampling uncertainty (unit-to-unit variability is large).",
    "The `cycle` feature lets the model use machine age; this helps on this benchmark but may not generalise.",
    "Explanations describe model behaviour, not physical causes (sensors are anonymized).",
    "The anomaly detector is unsupervised and evaluated only via a proxy (healthy vs near-failure rows).",
    "The out-of-distribution check is a simple range check.",
    f"Failure horizon H={FAILURE_HORIZON_H} is configured, not optimized; no sensitivity study over H was run.",
]
MODEL_CARD = {
    "model_version": MODEL_VERSION, "training_timestamp": TRAINING_TIMESTAMP, "evaluation_date": EVALUATION_DATE_UTC,
    "purpose": f"Estimate P(failure within the next {FAILURE_HORIZON_H} operating cycles) per machine and cycle, plus an anomaly score, a deterministic Machine Health Indicator (0-100), a data-quality status and a rule-based AI recommendation.",
    "dataset": {"name": f"NASA C-MAPSS {DATASET_ID}", "type": "simulated turbofan-engine run-to-failure benchmark", "sensors_anonymized": True,
                "note": "sensor_1..sensor_21 are anonymized; no physical meaning is assigned. Not motor/pump/compressor data.",
                "train_file_units": METADATA["dataset_info"]["train_file_units"], "train_file_rows": METADATA["dataset_info"]["train_file_rows"],
                "official_test_units": METADATA["dataset_info"]["official_test_units"], "official_test_rows": METADATA["dataset_info"]["official_test_rows"]},
    "target": {"name": "failure_within_h", "definition": "1 if remaining useful life <= H operating cycles else 0",
               "train_positive_rate": TRAIN_POSITIVE_RATE, "class_weighting_used": bool(USE_CLASS_WEIGHTS)},
    "failure_horizon": {"value": FAILURE_HORIZON_H, "unit": "operating_cycles", "status": "configured_not_optimized", "rationale": FAILURE_HORIZON_RATIONALE},
    "features": {"count": len(FEATURE_NAMES), "groups": ["raw sensors", "operating settings", "cycle", "rolling mean/std", "diff", "pct change", "slope"],
                 "active_sensors": ACTIVE_SENSORS, "active_operating_settings": ACTIVE_SETTINGS, "dropped_low_variance": DROPPED_SENSORS + DROPPED_SETTINGS,
                 "causal": "every feature uses only current and past cycles of the same unit (verified by test)"},
    "training_methodology": {"split": f"by unit_id: {len(train_units)}/{len(val_units)}/{len(test_units)} train/validation/internal-test; official test file kept as second unseen set",
                             "candidates": list(candidates.keys()), "selection_rule": "highest validation PR-AUC within tolerance, tie-break lowest Brier",
                             "selected_model": PRIMARY_NAME, "calibration_method": CALIBRATION_METHOD,
                             "calibration_selection": "grouped out-of-fold Brier on validation units",
                             "anomaly_detector": f"IsolationForest on healthy training rows (RUL >= {HEALTHY_MIN_RUL}); alarm threshold recalibrated on healthy validation rows",
                             "threshold": {"value": SELECTED_THRESHOLD, "rule": THRESHOLD_RULE, "selected_on": "validation units (out-of-fold calibrated probabilities)"}},
    "evaluation_methodology": ["metrics on unseen internal-test units", "metrics on official test_FD001 (all rows and last cycle per engine)",
                               "calibration via Brier score, ECE and reliability bins", "threshold analysis on validation and test", "leakage tests (causality, shuffled-label control)"],
    "metrics": {"validation_oof_calibrated": mi, "internal_test": mt, "official_test_all_rows": mo,
                "official_test_last_cycle_per_unit": METRICS["official_test_last_cycle_per_unit"]},
    "anomaly_detection": {"healthy_flag_rates": ANOMALY_FLAG_RATES, "proxy_evaluation": METRICS["anomaly_detection_proxy"]},
    "health_indicator": {"trained_model": False, "formula": METADATA["health_indicator"]["formula"], "config": HEALTH_CONFIG,
                         "additive_rows": ["failure_risk", "anomaly", "data_quality"], "trend": "not enabled"},
    "recommendation": {"label": RECOMMENDATION_CONFIG["label"], "uses_llm": False, "config": RECOMMENDATION_CONFIG},
    "data_quality_mapping": {"DATA_OK": "reliable", "DATA_WARNING": "reliability_reduced", "DATA_INVALID": "not_scored"},
    "prediction_interpretation": [
        "failure_probability is a calibrated ESTIMATE, not a guarantee; calibration quality is measured, not assured.",
        "anomaly_score is a percentile rank versus healthy reference data, not a probability; an anomaly is not a failure.",
        "machine_health_indicator is a deterministic formula, not a trained model and not a physical measurement.",
        "data_quality_status is independent of the prediction; DATA_INVALID inputs are not scored.",
        "Recommendations are AI-generated, rule-based (no LLM) and are not a confirmed diagnosis."],
    "intended_use": ["Portfolio/MVP demonstration of an end-to-end predictive-maintenance ML pipeline", "Decision support on C-MAPSS-style multivariate run-to-failure data"],
    "non_intended_use": ["Safety-critical or real maintenance decisions", "Real aircraft engines", "Machine types or sensor layouts not matching the trained schema",
                         "Automated maintenance actions without human review"],
    "known_limitations": LIMITATIONS,
    "library_versions": library_versions,
}
dump_json(MODEL_CARD, A / "metadata" / "model_card.json")

model_card = f"""# Model Card — AI Predictive Maintenance: Failure-Within-H Model

**Version:** `{MODEL_VERSION}`  |  **Training timestamp (UTC):** {TRAINING_TIMESTAMP}  |  **Evaluation date (UTC):** {EVALUATION_DATE_UTC}

## Model purpose
{MODEL_CARD['purpose']}

## Dataset
NASA C-MAPSS **{DATASET_ID}** — a **simulated turbofan-engine run-to-failure benchmark**. Sensor columns `sensor_1`-`sensor_21` are **anonymized**; no physical meaning is assigned. It is not motor, pump or compressor data. Train file: {METADATA['dataset_info']['train_file_units']} units / {METADATA['dataset_info']['train_file_rows']:,} rows; official test: {METADATA['dataset_info']['official_test_units']} units / {METADATA['dataset_info']['official_test_rows']:,} rows.

## Target and failure horizon
`failure_within_h` = 1 if remaining useful life <= **H = {FAILURE_HORIZON_H} cycles**, else 0. Train positive rate: {TRAIN_POSITIVE_RATE:.3f}.
**Why H = {FAILURE_HORIZON_H}? (configured, not optimized)** {FAILURE_HORIZON_RATIONALE}

## Features ({len(FEATURE_NAMES)})
Per-unit **causal** features: raw active sensors ({len(ACTIVE_SENSORS)}), non-constant operating settings ({len(ACTIVE_SETTINGS)}), cycle number, rolling mean/std (windows {list(ROLLING_WINDOWS)}), 1-cycle difference and safe percentage change, and {SLOPE_WINDOW}-cycle slope. Dropped (constant/low variance in training units): {DROPPED_SENSORS + DROPPED_SETTINGS}.

## Training methodology
* Units split into train/validation/internal-test **by `unit_id`** ({len(train_units)}/{len(val_units)}/{len(test_units)}); the official test file is a second unseen set.
* Candidates: logistic regression, HistGradientBoosting{', XGBoost' if XGB_AVAILABLE else ''}. Selected by validation PR-AUC (tie-break: Brier) -> **{PRIMARY_NAME}**. Class weighting used: {USE_CLASS_WEIGHTS}.
* Probability calibration: **{CALIBRATION_METHOD}**, chosen by grouped out-of-fold Brier on validation units.
* Anomaly detector: Isolation Forest trained on healthy training rows (RUL >= {HEALTHY_MIN_RUL}); alarm threshold recalibrated on healthy validation rows (measured healthy flag rate on unseen internal-test units: {ANOMALY_FLAG_RATES['isolation_forest']['internal_test_healthy_rows']:.2%} vs target {ANOMALY_CONTAMINATION:.1%}).
* Decision threshold **{SELECTED_THRESHOLD:.2f}** ({THRESHOLD_RULE}).

## Metrics (measured in this run; threshold {SELECTED_THRESHOLD:.2f})
| Set | Precision | Recall | F1 | ROC-AUC | PR-AUC | Brier | ECE |
|---|---|---|---|---|---|---|---|
| Validation (out-of-fold calibrated) | {fmt(mi['precision'])} | {fmt(mi['recall'])} | {fmt(mi['f1'])} | {fmt(mi['roc_auc'])} | {fmt(mi['pr_auc'])} | {fmt(mi['brier'],4)} | {fmt(mi['ece'])} |
| Internal test (unseen units) | {fmt(mt['precision'])} | {fmt(mt['recall'])} | {fmt(mt['f1'])} | {fmt(mt['roc_auc'])} | {fmt(mt['pr_auc'])} | {fmt(mt['brier'],4)} | {fmt(mt['ece'])} |
| Official test_FD001 (all rows) | {fmt(mo['precision'])} | {fmt(mo['recall'])} | {fmt(mo['f1'])} | {fmt(mo['roc_auc'])} | {fmt(mo['pr_auc'])} | {fmt(mo['brier'],4)} | {fmt(mo['ece'])} |

Full metrics incl. accuracy: `evaluation/metrics.json`; curves and bins: `evaluation/curves.json`.

## Prediction interpretation
""" + "\n".join(f"* {t}" for t in MODEL_CARD["prediction_interpretation"]) + f"""

Data quality mapping: `DATA_OK` = reliable, `DATA_WARNING` = reliability reduced, `DATA_INVALID` = not scored.

## Intended use
""" + "\n".join(f"* {t}" for t in MODEL_CARD["intended_use"]) + """

## Non-intended use
""" + "\n".join(f"* {t}" for t in MODEL_CARD["non_intended_use"]) + """

## Known limitations
""" + "\n".join(f"* {t}" for t in LIMITATIONS) + "\n"
(A / "metadata" / "MODEL_CARD.md").write_text(model_card)
display(Markdown(model_card))

In [ ]:
def write_manifest():
    """SHA-256 of every artifact file (including code/*.py). Re-run whenever files change."""
    files = {}
    for p in sorted(A.rglob("*")):
        if p.is_file() and "__pycache__" not in p.parts and p.suffix != ".pyc" and p.name != "artifact_manifest.json":
            files[p.relative_to(A).as_posix()] = {"sha256": hashlib.sha256(p.read_bytes()).hexdigest(), "bytes": p.stat().st_size}
    dump_json({"model_version": MODEL_VERSION, "hash_algorithm": "sha256", "files": files}, A / "metadata" / "artifact_manifest.json")
    return len(files)

def load_verifier():
    spec = importlib.util.spec_from_file_location("verify_artifacts", A / "verify_artifacts.py")
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    return module

print(f"Manifest written for {write_manifest()} files.")

## 24. Artifact Reload Test (mandatory)

To prove the artifacts work **outside the training session**, a **fresh Python process** (no notebook variables, no in-memory models):

1. verifies every file against the manifest and checks library versions (`verify_artifacts.verify_all`),
2. only then adds `code/` to `sys.path` and loads everything with `ModelBundle.load`,
3. reads previously unseen official-test histories from a CSV and runs `predict_machine_state`,

and we compare its output with the in-memory results. We also check that **tampered artifacts and mismatching library versions are refused**. (Test inputs/outputs are written *outside* `artifacts/` so the manifest stays exact.)

In [ ]:
import subprocess
RELOAD_DIR = ARTIFACT_DIR.parent / "reload_test_tmp"; RELOAD_DIR.mkdir(exist_ok=True)
_ids = sorted(int(u) for u in test_raw.unit_id.unique())          # adapt to however many engines the uploaded dataset has
reload_units = [_ids[i] for i in sorted(set(np.linspace(0, len(_ids) - 1, min(6, len(_ids))).astype(int)))]
sample_csv, out_json = RELOAD_DIR / "reload_test_input.csv", RELOAD_DIR / "reload_test_output.json"
test_raw[test_raw.unit_id.isin(reload_units)].to_csv(sample_csv, index=False)

RELOAD_SCRIPT = r'''
import sys, json
from pathlib import Path
art = Path(sys.argv[1])
sys.path.insert(0, str(art))                      # test only: the backend VENDORS verify_artifacts instead
import verify_artifacts
verify_artifacts.verify_all(art)                  # SHA-256 of every file + library versions BEFORE importing artifact code
sys.path.insert(0, str(art / "code"))
import pandas as pd
from pdm_inference import ModelBundle, predict_machine_state
bundle = ModelBundle.load(art)
hist = pd.read_csv(sys.argv[2])
out = {str(int(u)): predict_machine_state(g, bundle) for u, g in hist.groupby("unit_id")}
Path(sys.argv[3]).write_text(json.dumps(out))
print("fresh process: verified + loaded", bundle.metadata["model_version"], "and scored", len(out), "machines")
'''
proc = subprocess.run([sys.executable, "-c", RELOAD_SCRIPT, str(A.resolve()), str(sample_csv), str(out_json)], capture_output=True, text=True)
print(proc.stdout.strip())
if proc.returncode != 0:
    print(proc.stderr); raise RuntimeError("Artifact reload test FAILED")

disk_results = json.loads(out_json.read_text())
max_diff = 0.0
for u in reload_units:
    mem = predict_machine_state(test_raw[test_raw.unit_id == u], bundle_mem)
    dsk = disk_results[str(u)]
    max_diff = max(max_diff, abs(mem["failure_probability"] - dsk["failure_probability"]), abs(mem["anomaly_raw_score"] - dsk["anomaly_raw_score"]))
    assert all(mem[k] == dsk[k] for k in ("prediction_status", "data_quality_status", "reliability", "machine_health_indicator"))
print(f"Reload test PASSED. Max absolute difference vs in-memory predictions: {max_diff:.2e}")
display(pd.DataFrame([{k: disk_results[str(u)][k] for k in ["machine_id", "cycle", "failure_probability", "anomaly_flag", "machine_health_indicator", "prediction_status", "reliability"]}
                      | {"recommendation": disk_results[str(u)]["maintenance_recommendation"]["category"]} for u in reload_units]))

# --- security / compatibility negative tests ---------------------------------------------------
verifier = load_verifier()
print("Manifest verification of the real artifacts:", verifier.verify_manifest(A))
tampered = RELOAD_DIR / "tampered"
shutil.rmtree(tampered, ignore_errors=True)
shutil.copytree(A, tampered, ignore=shutil.ignore_patterns("__pycache__"))
with open(tampered / "code" / "pdm_features.py", "a") as f:
    f.write("\n# tampered\n")
try:
    verifier.verify_manifest(tampered); raise AssertionError("modified code file was NOT detected")
except verifier.ArtifactVerificationError as e:
    print("  refused (modified code file):", str(e)[:90])
shutil.rmtree(tampered, ignore_errors=True)
shutil.copytree(A, tampered, ignore=shutil.ignore_patterns("__pycache__"))
(tampered / "code" / "evil.py").write_text("print('x')")
try:
    verifier.verify_manifest(tampered); raise AssertionError("unlisted code file was NOT detected")
except verifier.ArtifactVerificationError as e:
    print("  refused (unlisted extra code file):", str(e)[:90])
try:
    verifier.check_library_versions({"library_versions": {**library_versions, "scikit-learn": "0.0.1"}}); raise AssertionError("version mismatch not detected")
except verifier.ArtifactVerificationError as e:
    print("  refused (version mismatch):", str(e)[:90])
shutil.rmtree(tampered, ignore_errors=True)

## 25. FastAPI Integration Example

A future FastAPI service should (1) **verify the artifact folder** (manifest SHA-256 + library versions) using a *vendored* copy of `verify_artifacts.py`, (2) only then add `code/` to `sys.path` — or, better, **vendor `artifacts/code/pdm_*.py` into the backend** so no unverified code is imported at runtime — (3) load the bundle once at startup, and (4) call `predict_machine_state` for the latest state or `score_trajectory` for whole histories (vectorised; do **not** call `predict_machine_state` once per cycle). This is **not** a full application; it only demonstrates the interface and is not executed here — the same call path is simulated below.

In [ ]:
FASTAPI_EXAMPLE = r'''
"""Interface example only - NOT the production application."""
import sys
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

import verify_artifacts                     # VENDORED stdlib-only copy (do not import it from the artifact folder)

ARTIFACT_DIR = Path("artifacts")
state = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    # 1) refuse to start on tampered files or library-version mismatch
    verify_artifacts.verify_all(ARTIFACT_DIR)
    # 2) only now is it safe to import artifact code (alternatively: vendor pdm_*.py into the backend)
    sys.path.insert(0, str(ARTIFACT_DIR / "code"))
    from pdm_inference import ModelBundle
    state["bundle"] = ModelBundle.load(ARTIFACT_DIR)
    yield
    state.clear()


app = FastAPI(title="Predictive Maintenance inference (example)", lifespan=lifespan)


class PredictRequest(BaseModel):
    machine_id: Optional[Union[int, str]] = None
    history: List[Dict[str, Any]]      # chronological rows: cycle, operating settings, sensors


@app.get("/model/info")
def model_info():
    m = state["bundle"].metadata
    return {k: m[k] for k in ("model_version", "dataset", "failure_horizon", "failure_horizon_status", "calibration_method", "threshold", "training_timestamp")}


@app.post("/predict")
def predict(req: PredictRequest):
    from pdm_inference import predict_machine_state
    if not req.history:
        raise HTTPException(status_code=422, detail="history must not be empty")
    return predict_machine_state(pd.DataFrame(req.history), state["bundle"], machine_id=req.machine_id)


@app.post("/trajectory")
def trajectory(req: PredictRequest):
    from pdm_inference import score_trajectory, DataInvalidError
    try:
        return score_trajectory(pd.DataFrame(req.history), state["bundle"]).to_dict(orient="records")
    except DataInvalidError as e:
        raise HTTPException(status_code=422, detail=e.quality["issues"])
'''
(A / "fastapi_integration_example.py").write_text(FASTAPI_EXAMPLE)
write_manifest()                                              # the new file must be in the manifest before verifying

# Simulate the request path without running FastAPI: verify -> load -> JSON payload -> DataFrame -> predict
verifier = load_verifier()
meta_checked = verifier.verify_all(A)
bundle_loaded = ModelBundle.load(ARTIFACT_DIR)
_pu = _ids[min(6, len(_ids) - 1)]
payload = {"machine_id": f"ENGINE-{_pu:03d}", "history": json.loads(test_raw[test_raw.unit_id == _pu].to_json(orient="records"))}
response = predict_machine_state(pd.DataFrame(payload["history"]), bundle_loaded, machine_id=payload["machine_id"])
print(json.dumps({k: response[k] for k in ["machine_id", "cycle", "failure_probability", "failure_horizon", "anomaly_score", "anomaly_flag",
                                            "machine_health_indicator", "prediction_status", "data_quality_status", "reliability", "model_version"]}, indent=2))
print("Health breakdown:", response["health_breakdown"])
print("Top contributing features:", [(f["feature"], round(f["contribution"], 3), f["direction"]) for f in response["top_contributing_features"][:3]])
print("Recommendation:", response["maintenance_recommendation"]["label"], "->", response["maintenance_recommendation"]["category"])
traj = score_trajectory(pd.DataFrame(payload["history"]), bundle_loaded)
print(f"score_trajectory returned {len(traj)} per-cycle rows (vectorised); last status: {traj['prediction_status'].iloc[-1]}")

## 26. Final ML Summary

Everything below is printed from variables computed in this run — no values are typed in. The manifest is regenerated at the very end so it covers every file (including this summary), and the folder is zipped for download.

In [ ]:
mt, mo = METRICS["internal_test"], METRICS["official_test_all_rows"]
summary = f"""
================================ FINAL ML SUMMARY ================================
Dataset                : NASA C-MAPSS {DATASET_ID} (simulated turbofan benchmark; sensors anonymized)
Machines (train file)  : {train_raw.unit_id.nunique()}  -> train/val/internal-test units = {len(train_units)}/{len(val_units)}/{len(test_units)}
Machines (official test): {test_raw.unit_id.nunique()}
Observations           : train file {len(train_raw):,} rows | official test {len(test_raw):,} rows
Features used          : {len(FEATURE_NAMES)} ({len(ACTIVE_SENSORS)} active anonymized sensors, {len(ACTIVE_SETTINGS)} operating settings, cycle, rolling/diff/pct/slope features)
Failure horizon H      : {FAILURE_HORIZON_H} operating cycles - configured, NOT optimized   (train positive rate {TRAIN_POSITIVE_RATE:.3f}; class weighting: {USE_CLASS_WEIGHTS})
Anomaly detection      : IsolationForest (healthy rows RUL>={HEALTHY_MIN_RUL}), target flag rate {ANOMALY_CONTAMINATION:.1%}; healthy flag rate on unseen internal-test units {ANOMALY_FLAG_RATES['isolation_forest']['internal_test_healthy_rows']:.2%} (alarm threshold recalibrated on healthy validation rows)
Failure model          : {PRIMARY_NAME} ({type(final_model).__name__})
Calibration            : {CALIBRATION_METHOD}
Selected threshold     : {SELECTED_THRESHOLD:.2f}  ({THRESHOLD_RULE})
Internal-test metrics  : ROC-AUC {mt['roc_auc']:.3f} | PR-AUC {mt['pr_auc']:.3f} | Brier {mt['brier']:.4f} | ECE {mt['ece']:.3f} | precision {mt['precision']:.3f} | recall {mt['recall']:.3f} | F1 {mt['f1']:.3f} | accuracy {mt['accuracy']:.3f}
Official-test metrics  : ROC-AUC {mo['roc_auc']:.3f} | PR-AUC {mo['pr_auc']:.3f} | Brier {mo['brier']:.4f} | ECE {mo['ece']:.3f} | precision {mo['precision']:.3f} | recall {mo['recall']:.3f} | F1 {mo['f1']:.3f}
Model version          : {MODEL_VERSION}
Evaluation date (UTC)  : {EVALUATION_DATE_UTC}
Library versions       : {library_versions}
Demo engines (seed {DEMO_SEED}): {DEMO_EXPORT_UNITS}
Artifacts              : {ARTIFACT_DIR.resolve()}
Reload test            : verified + loaded in a fresh process (max diff vs in-memory {max_diff:.2e}); tampering and version mismatch are refused

Known limitations      : simulated single-condition dataset; few engines -> uncertain estimates; anonymized sensors (no physical
                         interpretation); H not optimized; health indicator is a formula, not a measurement; recommendations are
                         AI-generated, not a confirmed diagnosis; calibration and thresholds should be re-validated on any new data.
=================================================================================
"""
print(summary)
(A / "metadata" / "final_summary.txt").write_text(summary)

n_files = write_manifest()
verifier.verify_all(A)
print(f"Final manifest covers {n_files} files and verifies OK.")

import zipfile
zip_path = ARTIFACT_DIR.parent / "artifacts_bundle.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(A.rglob("*")):
        if p.is_file() and "__pycache__" not in p.parts and p.suffix != ".pyc":
            zf.write(p, Path("artifacts") / p.relative_to(A))
print("Zip created:", zip_path.resolve())
# In Colab you can download it with:  from google.colab import files; files.download("/content/artifacts_bundle.zip")